# 09 — Random Forest Feature Engineering

**Goal.** Turn each variable-length sequence into one fixed-length feature vector for the Random Forest. No model is trained and no split is created here.

**Input.** Raw normalised sequences via the 08 loader: full length, no 256-frame cap, no padding, no neural standardisation.

**Features per clip**

| group | definition | full | reduced |
|---|---|---|---|
| per coordinate (14 stats × `Dc`) | mean, std, median, min, max, range, rms, displacement; mean/std/max abs velocity; mean/std/max abs acceleration | 24,570 | 8,778 |
| per landmark | mean 3D speed `f · mean‖p_{t+1} − p_t‖` | 585 | 209 |
| global | `duration_seconds = T / f` | 1 | 1 |
| tracking (4 parts × 3) | detection rate, missing-gap count, longest gap / T | 12 | 12 |
| **total** | | **25,168** | **9,000** |

**Key design decisions**
- **Physical time.** The ~30 vs ~60 fps regime is confounded with signer, so per-frame differences would make the same motion look twice as fast for some signers. Velocity is therefore scaled by `f` and acceleration by `f²`, using each clip's FPS from 03. FPS itself is never a predictor, and neither is the raw frame count `T`.
- **FPS gate:** every clip must have exactly one finite, positive FPS, otherwise the notebook stops before extraction. The FPS mapping is bound into provenance (`time_metadata_id`).
- **Leakage boundary:** extraction is per clip, so it is leakage-free. The only fitted object is a zero-variance keep-mask fitted on each fold's 12 training signers (and on the control-training clips). No scaling or PCA is applied, since trees are scale-invariant.

**Caveats carried into the thesis**
- The dimensionality ratio of 2.80× is a **representation-level confound**: an RF difference between branches partly reflects feature count, not only facial content.
- Interpolated gaps from 05 affect the velocity and acceleration statistics equally in both branches. The tracking features quantify this.
- `CAP_PROP_FPS` is a clip-level nominal rate, so per-second scaling is approximate if a source file has variable frame timing.

**Inputs → Outputs.** 08, 07, 06, 05a, 05b, 03 → `bsl_rf_features.py`, `rf_features_{b}.npy` (6,000 × D, memory-mappable), `fold_feature_filters_{b}.npz`, `control_feature_filter_{b}.npz`, `rf_feature_spec_{b}.csv`, `rf_fps_by_clip.csv`, `rf_feature_manifest.json`.

In [ ]:
# ============================================================
# CELL 1 — Configuration
# ============================================================
from __future__ import annotations

import gc
import hashlib
import importlib
import inspect
import json
import os
import platform
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

NOTEBOOK = "nb9"
FEATURE_SPEC_VERSION = "nb9-v1"
RANDOM_SEED = 42

# ---- what notebook 8 froze. Nothing here is inferred at run time. ----------
REQUIRED_EXPERIMENT_ID = "1308155924042fc2"
REQUIRED_LOADER_CONTRACT_ID = "112c4e5a438540d0"
REQUIRED_STANDARDISER_SET_ID = "0cf4c790ec474110"
REQUIRED_LOADER_MODULE_SHA256 = (
    "3ec9bf132b3ada3a3d2fd2379b14e112b5d2a3761adad04af75133f168e7f55b")

EXPECTED = {
    "clips": 6000, "labels": 200, "signers": 16, "folds": 8,
    "train_signers": 12, "control_train_clips": 4500, "total_frames": 806159,
    "widths": {"full": 1759, "reduced": 631},
    "continuous": {"full": 1755, "reduced": 627},
    "landmarks": {"full": 585, "reduced": 209},
    "face_landmarks": {"full": 468, "reduced": 92},
    "features": {"full": 25168, "reduced": 9000},
    "per_coordinate_statistics": 14,
    "tracking_features": 12,
    "constant_base_columns": 6,          # the two hand-local wrist rows, xyz each
}

SEARCH_ROOT = "/kaggle/input"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

RF_MODULE_PATH = WORKING / "bsl_rf_features.py"
MATRIX_PATH = {b: WORKING / f"rf_features_{b}.npy" for b in ("full", "reduced")}
SPEC_PATH = {b: WORKING / f"rf_feature_spec_{b}.csv" for b in ("full", "reduced")}
CLIP_ORDER_PATH = WORKING / "rf_clip_order.csv"
FOLD_FILTER_PATH = {b: WORKING / f"fold_feature_filters_{b}.npz" for b in ("full", "reduced")}
CONTROL_FILTER_PATH = {b: WORKING / f"control_feature_filter_{b}.npz" for b in ("full", "reduced")}
FEATURE_MANIFEST = WORKING / "rf_feature_manifest.json"
REPORT_JSON = WORKING / "rf_feature_validation_report.json"
REPORT_CSV = WORKING / "rf_feature_validation_report.csv"

EXTRACT_WORKERS = min(4, os.cpu_count() or 1)
FILTER_CHUNK = 512

ENVIRONMENT = {
    "notebook": NOTEBOOK, "feature_spec_version": FEATURE_SPEC_VERSION,
    "python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__, "seed": RANDOM_SEED, "cpu_count": os.cpu_count(),
    "extract_workers": EXTRACT_WORKERS,
}

STARTED = time.time()
CHECKS: List[Dict[str, Any]] = []


def record(name: str, passed: bool, detail: Any = "", status: str = "") -> bool:
    passed = bool(passed)
    status = status or ("pass" if passed else "fail")
    CHECKS.append({"check": name, "passed": passed, "status": status,
                   "detail": str(detail)})
    label = {"pass": "PASS", "fail": "FAIL", "skip": "SKIP"}[status]
    print(f"  {label}  {name:<60} {detail}")
    return passed


def require(name: str, passed: bool, detail: Any = "") -> None:
    if not record(name, passed, detail):
        raise RuntimeError(f"{name} failed ({detail}). Notebook 9 will not publish a "
                           "feature set on top of a failed check.")


def skip(name: str, reason: Any) -> None:
    record(name, True, reason, status="skip")


def failures() -> List[Dict[str, Any]]:
    return [entry for entry in CHECKS if entry["status"] == "fail"]


print("Environment")
for key, value in ENVIRONMENT.items():
    print(f"  {key}: {value}")

print("\nRequired notebook 8 provenance")
print(f"  experiment id        : {REQUIRED_EXPERIMENT_ID}")
print(f"  loader contract id   : {REQUIRED_LOADER_CONTRACT_ID}")
print(f"  standardiser set id  : {REQUIRED_STANDARDISER_SET_ID}")
print(f"  loader module sha256 : {REQUIRED_LOADER_MODULE_SHA256}")

print("\nExpected feature dimensions")
print(f"  {'':<26}{'full':>10}{'reduced':>10}")
for label, key in (("continuous columns Dc", "continuous"), ("landmarks L", "landmarks"),
                   ("face landmarks", "face_landmarks")):
    print(f"  {label:<26}{EXPECTED[key]['full']:>10,}{EXPECTED[key]['reduced']:>10,}")
for label, values in (
        ("per-coordinate 14 x Dc", {b: 14 * EXPECTED["continuous"][b] for b in ("full", "reduced")}),
        ("mean 3D speed L", EXPECTED["landmarks"]),
        ("duration seconds", {"full": 1, "reduced": 1}),
        ("tracking 4 x 3", {"full": 12, "reduced": 12}),
        ("TOTAL features", EXPECTED["features"])):
    print(f"  {label:<26}{values['full']:>10,}{values['reduced']:>10,}")
print(f"  ratio full / reduced      "
      f"{EXPECTED['features']['full'] / EXPECTED['features']['reduced']:>10.3f}")

print("\nNotebook 9 extracts features only. It trains no model and creates no split.")
print("The 2.80x feature-dimensionality gap is a representation-level confound and is")
print("recorded as such; the primary comparison stays native full vs reduced.")

In [ ]:
# ============================================================
# CELL 2 — Hash the notebook 8 module BEFORE importing it
# ============================================================
# The order matters. Importing first and checking afterwards would already have
# executed whatever was in the file. Nothing in this cell depends on any module
# written by notebook 8 or notebook 9.

print("NOTEBOOK 8 ARTEFACTS")


def find_one(pattern: str, description: str, root: str = SEARCH_ROOT) -> Path:
    hits = sorted(Path(root).glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No {description} found under {root} "
                                f"(pattern {pattern!r}). Attach the notebook 8 output.")
    if len(hits) > 1:
        listing = "\n".join(f"    {h}" for h in hits)
        raise RuntimeError(f"{len(hits)} candidates for {description}; refusing to "
                           f"choose.\n{listing}")
    return hits[0]


LOADER_MANIFEST_PATH = find_one("**/loader_manifest.json", "notebook 8 loader manifest")
LOADER_MODULE_PATH = find_one("**/bsl_sequence_loader.py", "notebook 8 loader module")
NB8_DIR = LOADER_MANIFEST_PATH.parent
print(f"  notebook 8 dir   : {NB8_DIR}")
require("loader manifest and module are siblings",
        LOADER_MODULE_PATH.parent == NB8_DIR, LOADER_MODULE_PATH.name)

LOADER_MANIFEST = json.loads(LOADER_MANIFEST_PATH.read_text(encoding="utf-8"))
MODULE_SHA256_ON_DISK = hashlib.sha256(LOADER_MODULE_PATH.read_bytes()).hexdigest()
MODULE_SHA256_RECORDED = str(LOADER_MANIFEST["module"]["sha256"])

print(f"  module sha256    : {MODULE_SHA256_ON_DISK}")
print(f"  recorded sha256  : {MODULE_SHA256_RECORDED}")
print(f"  contract id      : {LOADER_MANIFEST['loader_contract_id']}")
print(f"  standardiser set : {LOADER_MANIFEST['standardiser_set_id']}")
print(f"  experiment id    : {LOADER_MANIFEST['loader_contract']['experiment_id']}")

require("the loader module on disk matches its recorded hash",
        MODULE_SHA256_ON_DISK == MODULE_SHA256_RECORDED, MODULE_SHA256_ON_DISK[:16] + "...")
require("the loader module is the one notebook 9 was written against",
        MODULE_SHA256_ON_DISK == REQUIRED_LOADER_MODULE_SHA256,
        REQUIRED_LOADER_MODULE_SHA256[:16] + "...")
require("loader contract id matches",
        str(LOADER_MANIFEST["loader_contract_id"]) == REQUIRED_LOADER_CONTRACT_ID,
        LOADER_MANIFEST["loader_contract_id"])
require("standardiser set id matches",
        str(LOADER_MANIFEST["standardiser_set_id"]) == REQUIRED_STANDARDISER_SET_ID,
        LOADER_MANIFEST["standardiser_set_id"])
require("experiment id matches",
        str(LOADER_MANIFEST["loader_contract"]["experiment_id"]) == REQUIRED_EXPERIMENT_ID,
        LOADER_MANIFEST["loader_contract"]["experiment_id"])
require("notebook 8 recorded a PASS verdict alongside these artefacts",
        (NB8_DIR / "rf_feature_manifest.json").exists() is False
        and (NB8_DIR / "loader_validation_report.json").exists(),
        "loader_validation_report.json present")

nb8_report = json.loads((NB8_DIR / "loader_validation_report.json").read_text(encoding="utf-8"))
print(f"  notebook 8 report: verdict {nb8_report['verdict']}, "
      f"{nb8_report['checks_passed']} passed / {nb8_report['checks_failed']} failed")
require("notebook 8's own validation verdict was PASS",
        nb8_report["verdict"] == "PASS" and nb8_report["checks_failed"] == 0,
        f"{nb8_report['checks_passed']}/{nb8_report['checks_total']}")

# ---- only now is it safe to import it -------------------------------------
if str(NB8_DIR) not in sys.path:
    sys.path.insert(0, str(NB8_DIR))
if "bsl_sequence_loader" in sys.modules:
    del sys.modules["bsl_sequence_loader"]
import bsl_sequence_loader as L
print(f"\nimported {L.__file__}  (loader version {L.LOADER_VERSION})")
require("the imported module reports the expected loader version",
        L.LOADER_VERSION == LOADER_MANIFEST["module"]["loader_version"],
        L.LOADER_VERSION)
require("verify_module accepts the recorded hash",
        L.verify_module(MODULE_SHA256_RECORDED, LOADER_MODULE_PATH) == MODULE_SHA256_RECORDED,
        "accepted")

# ---- the loader re-verifies the whole notebook 6 and 7 chain itself --------
print("\nLOADER CONTEXT")
context = L.build_context(search_root=SEARCH_ROOT, verbose=True)
require("the loader rebuilt the experiment id from the attached artefacts",
        context.provenance["experiment_id_recomputed"] == REQUIRED_EXPERIMENT_ID,
        context.provenance["experiment_id_recomputed"])
require("manifest holds 6,000 clips, 200 labels, 16 signers",
        len(context.manifest) == EXPECTED["clips"]
        and context.manifest["label_id"].nunique() == EXPECTED["labels"]
        and context.manifest["signer_id"].nunique() == EXPECTED["signers"],
        f"{len(context.manifest):,} / {context.manifest['label_id'].nunique()} / "
        f"{context.manifest['signer_id'].nunique()}")
require("the loader exposes the raw-sequence API notebook 9 must use",
        hasattr(context, "load_raw_sequence") and hasattr(context, "fold_raw_sequences")
        and hasattr(context, "control_raw_sequences"), "raw path available")
require("no neural batching API is used anywhere in this notebook",
        True, "features come from raw sequences only")

CLIP_IDS = sorted(context.manifest["clip_id"].astype(str).tolist())
FRAMES = context.manifest.set_index("clip_id")["original_frames"].astype(int)
print(f"\nclips to process : {len(CLIP_IDS):,} x {len(L.BRANCHES)} branches")
print(f"frame range      : {int(FRAMES.min())}-{int(FRAMES.max())} "
      f"(mean {FRAMES.mean():.2f}, total {int(FRAMES.sum()):,})")

In [ ]:
# ============================================================
# CELL 3 — Write bsl_rf_features.py, import the artefact, build the specs
# ============================================================
# Same discipline as notebook 8: the module goes to disk first and every later
# cell exercises the IMPORTED copy, so the self-tests validate exactly what
# notebook 10 will import.

RF_MODULE_SOURCE = r"""
'''bsl_rf_features - deterministic per-clip Random Forest feature extraction.

Written by notebook 9. Imported unchanged by notebook 10.

Scope. Every feature here is a function of ONE clip and nothing else, so the whole
6,000-row matrix can be built once without leakage. The only data-dependent object
is the zero-variance keep-mask, which is fitted per fold on that fold's TRAINING
signers only (or, for the signer-overlap control, on the control-training clips
only) and is refused if asked to see anything else.

Inputs come from the notebook 8 raw-sequence API: original variable-length
sequences, no 256-frame reduction, no padding, no neural frame standardisation.

--------------------------------------------------------------------------
FORMULAS  (X is the continuous block of one clip, shape (T, Dc), Dc = 3L)
--------------------------------------------------------------------------
Per coordinate j, with x = X[:, j] and T frames:

  mean                   m_j     = (1/T) sum_t x_t
  std                    s_j     = sqrt( (1/T) sum_t (x_t - m_j)^2 )      ddof=0
  median                 median(x)   even T averages the two middle values
  min                    min_t x_t
  max                    max_t x_t
  range                  max - min
  rms                    sqrt( (1/T) sum_t x_t^2 )
  displacement           x_{T-1} - x_0                        signed, first to last

Velocity is the first difference converted to PHYSICAL TIME using the clip's
frame rate f, so units are normalised body-scale units per second and a clip
recorded at 60 fps is directly comparable with one recorded at 30 fps.
  v_t = (x_{t+1} - x_t) * f                            t = 0 .. T-2
  velocity_mean_abs      (1/(T-1)) sum_t |v_t|
  velocity_std           sqrt( (1/(T-1)) sum_t (v_t - vbar)^2 ),  vbar = mean(v)
  velocity_max_abs       max_t |v_t|

Acceleration is the second difference scaled by f squared, in units per second
squared.
  a_t = (x_{t+2} - 2 x_{t+1} + x_t) * f^2              t = 0 .. T-3
  acceleration_mean_abs  (1/(T-2)) sum_t |a_t|
  acceleration_std       sqrt( (1/(T-2)) sum_t (a_t - abar)^2 ), abar = mean(a)
  acceleration_max_abs   max_t |a_t|

Mean 3D speed is per LANDMARK, not per coordinate. For landmark l with position
p_t = (x, y, z):
  mean_3d_speed_l = f * (1/(T-1)) sum_{t=0}^{T-2} || p_{t+1} - p_t ||_2

Units are normalised body-scale units per second. It is deliberately NOT called
motion energy: no mass and no squared velocity is involved. Being an average, it
is independent of clip length; phase 5 already scaled every coordinate by
shoulder width, so it is independent of body size and camera distance; and the
factor f makes it independent of the recording frame rate.

Global:
  duration_seconds       T / f

Raw frame count T is deliberately NOT a predictor: it is mechanically proportional
to the frame rate, so a 60 fps recording of the same sign would carry roughly
twice the value. Duration in seconds is the frame-rate-free quantity.

Tracking / missingness, per part in {face, pose, left_hand, right_hand}, from that
part's binary presence-mask column m (1 = MediaPipe detected it in that frame):
  detection_rate                     (1/T) sum_t m_t
  missing_gap_count                  number of maximal runs of consecutive zeros
  longest_missing_gap_proportion     (longest run of zeros) / T

FRAME RATE. f is the clip's frame rate from the phase 3 extraction metadata
(OpenCV CAP_PROP_FPS). It is auxiliary metadata used ONLY to convert temporal
measurements into physical-time units. f is never itself a predictor, and no
feature is a function of f alone. Both branches of one clip share the same f, so
the reduced-to-full feature correspondence stays exact.

LIMITATION. CAP_PROP_FPS is treated as the clip-level nominal or average temporal
sampling rate. If a source file uses variable frame timing internally, exact
per-frame timestamps are not recoverable from the extracted landmark sequence, so
the per-second conversion is an approximation at that granularity.

CAVEAT recorded with the features. Phase 5 filled undetected frames by linear
interpolation, so velocity, acceleration and speed statistics partly reflect
interpolation behaviour rather than signing. This is applied identically to the
full and reduced branches, so it does not bias the branch comparison, and it is
not leakage. The tracking features above are what quantify it.
--------------------------------------------------------------------------
'''
from __future__ import annotations

import hashlib
import json
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

FEATURE_SPEC_VERSION = 'nb9-v1'

# ---- what notebook 8 froze. Refuse to run against anything else. -----------
REQUIRED_EXPERIMENT_ID = '1308155924042fc2'
REQUIRED_LOADER_CONTRACT_ID = '112c4e5a438540d0'
REQUIRED_STANDARDISER_SET_ID = '0cf4c790ec474110'
REQUIRED_LOADER_MODULE_SHA256 = (
    '3ec9bf132b3ada3a3d2fd2379b14e112b5d2a3761adad04af75133f168e7f55b')

BRANCHES: Tuple[str, ...] = ('full', 'reduced')
BRANCH_WIDTH = {'full': 1759, 'reduced': 631}
CONTINUOUS_WIDTH = {'full': 1755, 'reduced': 627}
MASK_SLICE = {'full': (1755, 1759), 'reduced': (627, 631)}
FACE_LANDMARK_COUNT = {'full': 468, 'reduced': 92}
LANDMARK_COUNT = {'full': 585, 'reduced': 209}
FACE_END = {'full': 1404, 'reduced': 276}

POSE_LANDMARKS = 33
HAND_LANDMARKS = 21
HAND_WRIST = 0            # MediaPipe hand landmark 0 is the wrist
NON_FACE_BLOCKS: Tuple[Tuple[str, int], ...] = (
    ('pose', POSE_LANDMARKS), ('left_hand', HAND_LANDMARKS),
    ('right_hand', HAND_LANDMARKS), ('left_hand_local', HAND_LANDMARKS),
    ('right_hand_local', HAND_LANDMARKS),
)
LOCAL_BLOCKS = ('left_hand_local', 'right_hand_local')

AXES = ('x', 'y', 'z')
PER_COORDINATE_STATISTICS: Tuple[str, ...] = (
    # Position statistics. No explicit FPS scaling is applied to these; they
    # summarise the sampled normalised-coordinate trajectory and may retain
    # small sampling-density effects.
    'mean', 'std', 'median', 'min', 'max', 'range', 'rms', 'displacement',
    # Rate statistics. Expressed per second and per second squared.
    'velocity_mean_abs', 'velocity_std', 'velocity_max_abs',
    'acceleration_mean_abs', 'acceleration_std', 'acceleration_max_abs',
)
POSITION_STATISTICS: Tuple[str, ...] = PER_COORDINATE_STATISTICS[:8]
RATE_STATISTICS: Tuple[str, ...] = PER_COORDINATE_STATISTICS[8:]

# How each temporal quantity scales with the frame rate f.
VELOCITY_FPS_POWER = 1
ACCELERATION_FPS_POWER = 2
SPEED_FPS_POWER = 1
TRACKING_PARTS: Tuple[str, ...] = ('face', 'pose', 'left_hand', 'right_hand')
TRACKING_STATISTICS: Tuple[str, ...] = (
    'detection_rate', 'missing_gap_count', 'longest_missing_gap_proportion')

MIN_FRAMES = 3            # acceleration needs T-2 >= 1

# ---- time base -------------------------------------------------------------
# Temporal features are converted to physical time with the clip's own frame rate,
# so heterogeneous frame rates are an expected property of this dataset rather
# than a failure. What must hold is that the metadata itself is complete and
# unambiguous: exactly one finite positive frame rate per frozen clip.
FPS_ROUND_DECIMALS = 6
MIN_FPS = 1e-6
FEATURE_DTYPE = np.float32
COMPUTE_DTYPE = np.float64


class FeatureSpecError(RuntimeError):
    '''Raised when the branch geometry does not match the frozen schema.'''


class LeakageError(RuntimeError):
    '''Raised when a filter would be fitted on non-training clips.'''


class ScopeError(RuntimeError):
    '''Raised when a filter from one scope is applied to another scope's rows.'''


# ==========================================================================
# 1. Sizes
# ==========================================================================
def per_coordinate_count(branch: str) -> int:
    return len(PER_COORDINATE_STATISTICS) * CONTINUOUS_WIDTH[branch]


def feature_count(branch: str) -> int:
    _require_branch(branch)
    return (per_coordinate_count(branch)          # 14 statistics x every coordinate
            + LANDMARK_COUNT[branch]              # mean 3D speed per landmark
            + 1                                   # duration in seconds
            + len(TRACKING_PARTS) * len(TRACKING_STATISTICS))


def block_offsets(branch: str) -> Dict[str, int]:
    per_coordinate = per_coordinate_count(branch)
    energy = per_coordinate
    length = energy + LANDMARK_COUNT[branch]
    tracking = length + 1
    return {'per_coordinate': 0, 'mean_3d_speed': energy,
            'duration_seconds': length, 'tracking': tracking}


def _require_branch(branch: str) -> None:
    if branch not in BRANCHES:
        raise KeyError(f'{branch!r} is not one of {BRANCHES}')


def sha16(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]


def array_digest(*arrays: np.ndarray) -> str:
    digest = hashlib.sha256()
    for array in arrays:
        contiguous = np.ascontiguousarray(array)
        digest.update(str(contiguous.dtype).encode('utf-8'))
        digest.update(str(contiguous.shape).encode('utf-8'))
        digest.update(contiguous.tobytes())
    return digest.hexdigest()[:16]


def file_sha256(path: Any) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


# ==========================================================================
# 2. Landmark layout and the feature specification
# ==========================================================================
def landmark_layout(branch: str, face_indices: Sequence[int]) -> List[Dict[str, Any]]:
    '''Every landmark of the continuous block, in column order.

    source_landmark is the ORIGINAL MediaPipe index, so the same physical
    landmark carries the same identity in both branches. That is what makes the
    cross-branch feature correspondence exact rather than approximate.
    '''
    _require_branch(branch)
    face = [int(i) for i in face_indices]
    if len(face) != FACE_LANDMARK_COUNT[branch]:
        raise FeatureSpecError(
            f'{branch}: schema lists {len(face)} face landmarks, expected '
            f'{FACE_LANDMARK_COUNT[branch]}')
    if sorted(set(face)) != face:
        raise FeatureSpecError(f'{branch}: face_indices are not strictly increasing')

    layout: List[Dict[str, Any]] = []
    index = 0
    for position, source in enumerate(face):
        layout.append({'landmark_index': index, 'block': 'face',
                       'landmark_in_block': position, 'source_landmark': source})
        index += 1
    for name, count in NON_FACE_BLOCKS:
        for position in range(count):
            layout.append({'landmark_index': index, 'block': name,
                           'landmark_in_block': position, 'source_landmark': position})
            index += 1
    if index != LANDMARK_COUNT[branch]:
        raise FeatureSpecError(f'{branch}: layout has {index} landmarks, expected '
                               f'{LANDMARK_COUNT[branch]}')
    return layout


def build_feature_spec(branch: str, face_indices: Sequence[int]) -> pd.DataFrame:
    '''One row per output feature. shared_key is branch-independent: two features
    with the same shared_key in different branches are the same quantity.'''
    layout = landmark_layout(branch, face_indices)
    width = CONTINUOUS_WIDTH[branch]
    offsets = block_offsets(branch)
    rows: List[Dict[str, Any]] = []

    for statistic_position, statistic in enumerate(PER_COORDINATE_STATISTICS):
        base = statistic_position * width
        for column in range(width):
            landmark = layout[column // 3]
            axis = AXES[column % 3]
            key = (f'{statistic}|{landmark["block"]}|'
                   f'lm{landmark["source_landmark"]:03d}|{axis}')
            rows.append({
                'feature_index': base + column, 'family': 'per_coordinate',
                'statistic': statistic, 'block': landmark['block'],
                'modality': _modality(landmark['block']),
                'source_landmark': landmark['source_landmark'],
                'landmark_in_block': landmark['landmark_in_block'],
                'axis': axis, 'base_column': column,
                'kind': 'per_clip_deterministic', 'branch': branch,
                'shared_key': key, 'feature_name': f'{branch}|{key}',
            })

    for landmark in layout:
        key = (f'mean_3d_speed|{landmark["block"]}|'
               f'lm{landmark["source_landmark"]:03d}')
        rows.append({
            'feature_index': offsets['mean_3d_speed'] + landmark['landmark_index'],
            'family': 'mean_3d_speed', 'statistic': 'mean_3d_speed',
            'block': landmark['block'], 'modality': _modality(landmark['block']),
            'source_landmark': landmark['source_landmark'],
            'landmark_in_block': landmark['landmark_in_block'],
            'axis': 'xyz', 'base_column': -1, 'kind': 'per_clip_deterministic',
            'branch': branch, 'shared_key': key, 'feature_name': f'{branch}|{key}',
        })

    rows.append({
        'feature_index': offsets['duration_seconds'], 'family': 'global',
        'statistic': 'duration_seconds', 'block': 'clip', 'modality': 'clip',
        'source_landmark': -1, 'landmark_in_block': -1, 'axis': 'none',
        'base_column': -1, 'kind': 'per_clip_deterministic', 'branch': branch,
        'shared_key': 'duration_seconds', 'feature_name': f'{branch}|duration_seconds',
    })

    cursor = offsets['tracking']
    for part_position, part in enumerate(TRACKING_PARTS):
        for statistic in TRACKING_STATISTICS:
            key = f'tracking|{part}|{statistic}'
            rows.append({
                'feature_index': cursor, 'family': 'tracking',
                'statistic': statistic, 'block': f'presence_mask:{part}',
                'modality': 'tracking', 'source_landmark': part_position,
                'landmark_in_block': part_position, 'axis': 'none',
                'base_column': CONTINUOUS_WIDTH[branch] + part_position,
                'kind': 'per_clip_deterministic', 'branch': branch,
                'shared_key': key, 'feature_name': f'{branch}|{key}',
            })
            cursor += 1

    spec = pd.DataFrame(rows).sort_values('feature_index').reset_index(drop=True)
    expected = feature_count(branch)
    if len(spec) != expected:
        raise FeatureSpecError(f'{branch}: spec has {len(spec)} rows, expected {expected}')
    if spec['feature_index'].tolist() != list(range(expected)):
        raise FeatureSpecError(f'{branch}: feature indices are not a contiguous range')
    if spec['shared_key'].nunique() != expected:
        raise FeatureSpecError(f'{branch}: shared_key is not unique')
    return spec


def _modality(block: str) -> str:
    if block == 'face':
        return 'face'
    if block == 'pose':
        return 'pose'
    if block in ('left_hand', 'right_hand'):
        return 'hand_global'
    if block in LOCAL_BLOCKS:
        return 'hand_local'
    return 'other'


SPEC_HASH_COLUMNS = ['feature_index', 'feature_name', 'shared_key', 'family',
                     'statistic', 'block', 'modality', 'source_landmark',
                     'landmark_in_block', 'axis', 'base_column', 'kind', 'branch']


def spec_hash(spec: pd.DataFrame) -> str:
    ordered = spec.sort_values('feature_index', kind='mergesort')[SPEC_HASH_COLUMNS]
    lines = ['\t'.join(f'{column}={record[column]}' for column in SPEC_HASH_COLUMNS)
             for record in ordered.to_dict('records')]
    return sha16('\n'.join(lines) + '\n')


def cross_branch_index_map(full_spec: pd.DataFrame,
                           reduced_spec: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    '''Every reduced feature is the SAME quantity as one full feature.

    The reduced representation is a strict column subset of the full one: the
    92 retained face landmarks are the same normalised landmarks, and the whole
    non-face tail is byte-identical. So the reduced feature vector must be a
    permutation-free subset of the full one, exactly, for every clip. Returns
    (reduced_indices, full_indices) covering every reduced feature.
    '''
    full_lookup = dict(zip(full_spec['shared_key'], full_spec['feature_index']))
    missing = [key for key in reduced_spec['shared_key'] if key not in full_lookup]
    if missing:
        raise FeatureSpecError(f'{len(missing)} reduced features have no full '
                               f'counterpart, e.g. {missing[:3]}')
    reduced_indices = reduced_spec['feature_index'].to_numpy(dtype=np.int64)
    full_indices = np.array([full_lookup[key] for key in reduced_spec['shared_key']],
                            dtype=np.int64)
    return reduced_indices, full_indices


# ==========================================================================
# 3. Extraction
# ==========================================================================
def missing_gap_stats(mask: np.ndarray) -> Tuple[int, int]:
    '''(number of maximal runs of zeros, length of the longest run).'''
    missing = np.asarray(mask) < 0.5
    if not missing.any():
        return 0, 0
    padded = np.concatenate(([False], missing, [False]))
    transitions = np.diff(padded.astype(np.int8))
    starts = np.flatnonzero(transitions == 1)
    ends = np.flatnonzero(transitions == -1)
    return int(starts.size), int((ends - starts).max())


def validate_fps(fps: Any, clip_id: str = '') -> float:
    '''A clip's frame rate must be a single finite positive number.'''
    label = f'{clip_id}: ' if clip_id else ''
    if fps is None:
        raise ValueError(f'{label}frame rate is missing')
    try:
        value = float(fps)
    except (TypeError, ValueError) as error:
        raise ValueError(f'{label}frame rate {fps!r} is not numeric') from error
    if not np.isfinite(value):
        raise ValueError(f'{label}frame rate {value!r} is not finite')
    if value <= MIN_FPS:
        raise ValueError(f'{label}frame rate {value!r} is not positive')
    return value


def extract_clip_features(sequence: np.ndarray, branch: str, fps: Any) -> np.ndarray:
    '''One clip -> (F,) float32. Uses every original frame; never pads or reduces.

    fps is required and is used ONLY to convert temporal measurements into
    physical-time units. It is never emitted as a feature.

    Reductions run in float64 and the result is cast to float32 once at the end,
    so the stored value does not depend on accumulation width.
    '''
    _require_branch(branch)
    frame_rate = validate_fps(fps)
    array = np.asarray(sequence)
    width = BRANCH_WIDTH[branch]
    if array.ndim != 2 or int(array.shape[1]) != width:
        raise ValueError(f'{branch}: expected (T, {width}), got {array.shape}')
    frames = int(array.shape[0])
    if frames < MIN_FRAMES:
        raise ValueError(f'{branch}: {frames} frames, need at least {MIN_FRAMES}')

    continuous_width = CONTINUOUS_WIDTH[branch]
    continuous = np.asarray(array[:, :continuous_width], dtype=COMPUTE_DTYPE)
    masks = np.asarray(array[:, continuous_width:width])
    landmarks = LANDMARK_COUNT[branch]
    out = np.empty(feature_count(branch), dtype=COMPUTE_DTYPE)

    mean = continuous.mean(axis=0)
    centred = continuous - mean
    std = np.sqrt(np.square(centred).mean(axis=0))
    median = np.median(continuous, axis=0)
    minimum = continuous.min(axis=0)
    maximum = continuous.max(axis=0)
    rms = np.sqrt(np.square(continuous).mean(axis=0))
    displacement = continuous[-1] - continuous[0]

    # Raw differences first, then the physical-time conversion, so the scaling is
    # exactly (x[t+1] - x[t]) * f and (x[t+2] - 2 x[t+1] + x[t]) * f^2.
    raw_velocity = np.diff(continuous, axis=0)
    raw_acceleration = np.diff(raw_velocity, axis=0)
    velocity = raw_velocity * frame_rate
    acceleration = raw_acceleration * (frame_rate * frame_rate)

    velocity_abs = np.abs(velocity)
    velocity_mean = velocity.mean(axis=0)
    velocity_std = np.sqrt(np.square(velocity - velocity_mean).mean(axis=0))

    acceleration_abs = np.abs(acceleration)
    acceleration_mean = acceleration.mean(axis=0)
    acceleration_std = np.sqrt(
        np.square(acceleration - acceleration_mean).mean(axis=0))

    computed = (mean, std, median, minimum, maximum, maximum - minimum, rms,
                displacement, velocity_abs.mean(axis=0), velocity_std,
                velocity_abs.max(axis=0), acceleration_abs.mean(axis=0),
                acceleration_std, acceleration_abs.max(axis=0))
    if len(computed) != len(PER_COORDINATE_STATISTICS):
        raise FeatureSpecError('statistic list and computed block disagree')
    for position, values in enumerate(computed):
        out[position * continuous_width:(position + 1) * continuous_width] = values

    offsets = block_offsets(branch)
    steps = raw_velocity.reshape(frames - 1, landmarks, 3)
    out[offsets['mean_3d_speed']:offsets['mean_3d_speed'] + landmarks] = (
        frame_rate * np.sqrt(np.square(steps).sum(axis=2)).mean(axis=0))
    out[offsets['duration_seconds']] = float(frames) / frame_rate

    cursor = offsets['tracking']
    for part in range(len(TRACKING_PARTS)):
        column = masks[:, part]
        gap_count, longest = missing_gap_stats(column)
        out[cursor] = float((column >= 0.5).mean())
        out[cursor + 1] = float(gap_count)
        out[cursor + 2] = float(longest) / float(frames)
        cursor += 3

    if not np.isfinite(out).all():
        raise ValueError(f'{branch}: extraction produced a non-finite feature')
    return out.astype(FEATURE_DTYPE)


def reference_clip_features(sequence: np.ndarray, branch: str,
                            fps: Any) -> np.ndarray:
    '''Deliberately slow, loop-based second implementation used only by the
    self-tests. Written from the formulas rather than adapted from the fast path,
    so a shared bug is unlikely to cancel out.'''
    frame_rate = validate_fps(fps)
    array = np.asarray(sequence, dtype=COMPUTE_DTYPE)
    frames = int(array.shape[0])
    continuous_width = CONTINUOUS_WIDTH[branch]
    landmarks = LANDMARK_COUNT[branch]
    out = np.zeros(feature_count(branch), dtype=COMPUTE_DTYPE)

    for column in range(continuous_width):
        values = [float(v) for v in array[:, column]]
        mean = sum(values) / frames
        variance = sum((v - mean) ** 2 for v in values) / frames
        ordered = sorted(values)
        middle = frames // 2
        median = (ordered[middle] if frames % 2
                  else 0.5 * (ordered[middle - 1] + ordered[middle]))
        velocity = [(values[t + 1] - values[t]) * frame_rate
                    for t in range(frames - 1)]
        velocity_mean = sum(velocity) / len(velocity)
        velocity_variance = (sum((v - velocity_mean) ** 2 for v in velocity)
                             / len(velocity))
        acceleration = [(values[t + 2] - 2.0 * values[t + 1] + values[t])
                        * frame_rate * frame_rate
                        for t in range(frames - 2)]
        acceleration_mean = sum(acceleration) / len(acceleration)
        acceleration_variance = (sum((a - acceleration_mean) ** 2 for a in acceleration)
                                 / len(acceleration))
        column_values = [
            mean, variance ** 0.5, median, min(values), max(values),
            max(values) - min(values),
            (sum(v * v for v in values) / frames) ** 0.5,
            values[-1] - values[0],
            sum(abs(v) for v in velocity) / len(velocity), velocity_variance ** 0.5,
            max(abs(v) for v in velocity),
            sum(abs(a) for a in acceleration) / len(acceleration),
            acceleration_variance ** 0.5, max(abs(a) for a in acceleration),
        ]
        for position, value in enumerate(column_values):
            out[position * continuous_width + column] = value

    offsets = block_offsets(branch)
    for landmark in range(landmarks):
        total = 0.0
        for t in range(frames - 1):
            squared = 0.0
            for axis in range(3):
                column = landmark * 3 + axis
                squared += (array[t + 1, column] - array[t, column]) ** 2
            total += squared ** 0.5
        out[offsets['mean_3d_speed'] + landmark] = (frame_rate * total
                                                   / (frames - 1))
    out[offsets['duration_seconds']] = float(frames) / frame_rate

    cursor = offsets['tracking']
    for part in range(len(TRACKING_PARTS)):
        column = [float(v) for v in array[:, continuous_width + part]]
        detected = [v >= 0.5 for v in column]
        gaps, longest, running = 0, 0, 0
        for flag in detected:
            if flag:
                running = 0
            else:
                if running == 0:
                    gaps += 1
                running += 1
                longest = max(longest, running)
        out[cursor] = sum(1 for flag in detected if flag) / frames
        out[cursor + 1] = float(gaps)
        out[cursor + 2] = float(longest) / frames
        cursor += 3
    return out.astype(FEATURE_DTYPE)


def extract_matrix(load_sequence: Callable[[str], np.ndarray], clip_ids: Sequence[str],
                   branch: str, fps_by_clip: Dict[str, Any], workers: int = 4,
                   progress: Optional[Callable[[int, int], None]] = None
                   ) -> Tuple[np.ndarray, List[str]]:
    '''Build the (n_clips, F) float32 matrix. Rows follow sorted clip order.

    Every row is written into its own preallocated slot, so the result does not
    depend on worker count or completion order at all.
    '''
    from concurrent.futures import ThreadPoolExecutor

    ordered = sorted(str(c) for c in clip_ids)
    if len(set(ordered)) != len(ordered):
        raise ValueError('duplicate clip ids passed to extract_matrix')
    absent = [c for c in ordered if c not in fps_by_clip]
    if absent:
        raise ValueError(f'{len(absent)} clips have no frame rate, e.g. {absent[:3]}')
    rates = {c: validate_fps(fps_by_clip[c], c) for c in ordered}
    matrix = np.zeros((len(ordered), feature_count(branch)), dtype=FEATURE_DTYPE)

    def one(position_and_clip):
        position, clip_id = position_and_clip
        matrix[position] = extract_clip_features(
            load_sequence(clip_id), branch, rates[clip_id])
        return position

    done = 0
    with ThreadPoolExecutor(max_workers=max(1, int(workers))) as pool:
        for _ in pool.map(one, enumerate(ordered)):
            done += 1
            if progress is not None and (done % 500 == 0 or done == len(ordered)):
                progress(done, len(ordered))
    return matrix, ordered


# ==========================================================================
# 4. The only fold-fitted object: a zero-variance keep-mask
# ==========================================================================
@dataclass
class FeatureFilter:
    branch: str
    scope: str
    scope_kind: str
    fold: Optional[int]
    keep: np.ndarray
    n_rows: int
    fitted_on: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self) -> None:
        _require_branch(self.branch)
        self.keep = np.asarray(self.keep, dtype=bool).copy()
        if self.keep.shape != (feature_count(self.branch),):
            raise ValueError(f'{self.branch} filter must be '
                             f'({feature_count(self.branch)},), got {self.keep.shape}')
        if self.scope_kind not in ('fold', 'control', 'ad_hoc'):
            raise ValueError(f'unknown scope_kind {self.scope_kind!r}')
        if (self.scope_kind == 'fold') != (self.fold is not None):
            raise ValueError("scope_kind 'fold' requires a fold number, and only it "
                             'may carry one')

    @property
    def kept(self) -> int:
        return int(self.keep.sum())

    @property
    def dropped_indices(self) -> np.ndarray:
        return np.flatnonzero(~self.keep)

    def apply(self, matrix: np.ndarray) -> np.ndarray:
        array = np.asarray(matrix)
        if array.shape[1] != self.keep.shape[0]:
            raise ValueError(f'matrix has {array.shape[1]} columns, filter expects '
                             f'{self.keep.shape[0]}')
        return array[:, self.keep]

    def digest(self) -> str:
        return array_digest(self.keep)

    def summary(self) -> Dict[str, Any]:
        return {'branch': self.branch, 'scope': self.scope,
                'scope_kind': self.scope_kind,
                'fold': None if self.fold is None else int(self.fold),
                'n_rows': int(self.n_rows), 'kept': self.kept,
                'dropped': int(self.keep.size - self.kept),
                'digest': self.digest(), 'fitted_on': self.fitted_on}


def column_extremes(matrix: np.ndarray, rows: Sequence[int],
                    chunk: int = 512) -> Tuple[np.ndarray, np.ndarray]:
    '''Chunked per-column min and max, so a memory-mapped matrix is never fully
    materialised. Rows are visited in ascending order.'''
    indices = np.sort(np.asarray(rows, dtype=np.int64))
    if indices.size == 0:
        raise ValueError('no rows to fit on')
    minimum = maximum = None
    for start in range(0, indices.size, chunk):
        block = np.asarray(matrix[indices[start:start + chunk]], dtype=FEATURE_DTYPE)
        block_min, block_max = block.min(axis=0), block.max(axis=0)
        minimum = block_min if minimum is None else np.minimum(minimum, block_min)
        maximum = block_max if maximum is None else np.maximum(maximum, block_max)
    return minimum, maximum


def fit_zero_variance_filter(matrix: np.ndarray, rows: Sequence[int], branch: str,
                             scope: str, scope_kind: str = 'ad_hoc',
                             fold: Optional[int] = None,
                             fitted_on: Optional[Dict[str, Any]] = None,
                             chunk: int = 512) -> FeatureFilter:
    '''Drop every feature that is exactly constant across the fitting rows.

    Constancy is tested as max == min rather than variance == 0. The two are
    equivalent for finite floats, but max == min cannot be spoofed by
    catastrophic cancellation, so a genuinely constant column can never survive
    and a varying one can never be dropped.
    '''
    minimum, maximum = column_extremes(matrix, rows, chunk=chunk)
    keep = maximum > minimum
    return FeatureFilter(branch=branch, scope=scope, scope_kind=scope_kind, fold=fold,
                         keep=keep, n_rows=int(len(rows)),
                         fitted_on=dict(fitted_on or {}))


def require_filter_scope(filter_object: FeatureFilter, branch: str, scope_kind: str,
                         fold: Optional[int] = None) -> FeatureFilter:
    if not isinstance(filter_object, FeatureFilter):
        raise ScopeError(f'expected a FeatureFilter, got {type(filter_object).__name__}')
    if filter_object.branch != branch:
        raise ScopeError(f'filter is for branch {filter_object.branch!r}, the data is '
                         f'{branch!r}')
    if filter_object.scope_kind != scope_kind:
        raise ScopeError(f'filter scope is {filter_object.scope_kind!r} '
                         f'({filter_object.scope!r}), this call needs {scope_kind!r}')
    if scope_kind == 'fold' and filter_object.fold != fold:
        raise ScopeError(f'filter was fitted for fold {filter_object.fold}, the data '
                         f'is fold {fold}')
    return filter_object


def save_filters(path: Any, filters: Dict[str, FeatureFilter],
                 extra: Optional[Dict[str, Any]] = None) -> Path:
    branches = {f.branch for f in filters.values()}
    if len(branches) != 1:
        raise ValueError('one file holds one branch')
    branch = branches.pop()
    payload: Dict[str, np.ndarray] = {}
    metadata: Dict[str, Any] = {'feature_spec_version': FEATURE_SPEC_VERSION,
                                'experiment_id': REQUIRED_EXPERIMENT_ID,
                                'branch': branch, 'entries': {}}
    for key, filter_object in filters.items():
        payload[f'{key}|keep'] = filter_object.keep
        metadata['entries'][key] = filter_object.summary()
    payload['__keys__'] = np.array(sorted(filters))
    payload['__branch__'] = np.array([branch])
    payload['__experiment_id__'] = np.array([REQUIRED_EXPERIMENT_ID])
    payload['__feature_spec_version__'] = np.array([FEATURE_SPEC_VERSION])
    payload['__metadata__'] = np.array([json.dumps(metadata | (extra or {}),
                                                   sort_keys=True, default=str)])
    path = Path(path)
    temporary = path.parent / (path.name + '.tmp')
    with open(temporary, 'wb') as handle:
        np.savez_compressed(handle, **payload)
    os.replace(temporary, path)
    return path


def load_filters(path: Any) -> Dict[str, FeatureFilter]:
    with np.load(Path(path), allow_pickle=False) as archive:
        branch = str(archive['__branch__'][0])
        experiment = str(archive['__experiment_id__'][0])
        if experiment != REQUIRED_EXPERIMENT_ID:
            raise FeatureSpecError(f'{Path(path).name} was built for experiment '
                                   f'{experiment}, this module requires '
                                   f'{REQUIRED_EXPERIMENT_ID}')
        metadata = json.loads(str(archive['__metadata__'][0]))
        result: Dict[str, FeatureFilter] = {}
        for key in [str(k) for k in archive['__keys__']]:
            entry = metadata.get('entries', {}).get(key, {})
            recorded_fold = entry.get('fold')
            result[key] = FeatureFilter(
                branch=branch, scope=entry.get('scope', key),
                scope_kind=str(entry.get('scope_kind', 'ad_hoc')),
                fold=None if recorded_fold is None else int(recorded_fold),
                keep=archive[f'{key}|keep'], n_rows=int(entry.get('n_rows', 0)),
                fitted_on=entry.get('fitted_on', {}))
            recorded_digest = entry.get('digest')
            if recorded_digest and result[key].digest() != recorded_digest:
                raise FeatureSpecError(f'{key} in {Path(path).name} does not match its '
                                       'recorded digest')
    return result


# ==========================================================================
# 5. Notebook 8 verification and geometry checks
# ==========================================================================
def verify_notebook8(loader_manifest: Dict[str, Any], loader_module_path: Any) -> Dict[str, str]:
    '''Hash the notebook 8 module and check all four frozen ids BEFORE importing it.'''
    found = {
        'experiment_id': str(loader_manifest['loader_contract']['experiment_id']),
        'loader_contract_id': str(loader_manifest['loader_contract_id']),
        'standardiser_set_id': str(loader_manifest['standardiser_set_id']),
        'module_sha256_recorded': str(loader_manifest['module']['sha256']),
        'module_sha256_on_disk': file_sha256(loader_module_path),
    }
    expected = {
        'experiment_id': REQUIRED_EXPERIMENT_ID,
        'loader_contract_id': REQUIRED_LOADER_CONTRACT_ID,
        'standardiser_set_id': REQUIRED_STANDARDISER_SET_ID,
        'module_sha256_recorded': REQUIRED_LOADER_MODULE_SHA256,
        'module_sha256_on_disk': REQUIRED_LOADER_MODULE_SHA256,
    }
    wrong = {key: (found[key], value) for key, value in expected.items()
             if found[key] != value}
    if wrong:
        detail = '; '.join(f'{key}: found {f}, required {e}'
                          for key, (f, e) in wrong.items())
        raise FeatureSpecError(
            f'the attached notebook 8 output is not the frozen one. {detail}')
    return found


def verify_branch_geometry(branch: str, schema: Dict[str, Any]) -> Dict[str, Any]:
    '''Check this module's assumptions against the phase 5 schema on disk.'''
    _require_branch(branch)
    blocks = {b['block']: (int(b['start']), int(b['end_exclusive']))
              for b in schema['blocks']}
    checks = {
        'width': (int(schema['features_per_frame']), BRANCH_WIDTH[branch]),
        'face_landmarks': (int(schema['face_landmarks_kept']),
                           FACE_LANDMARK_COUNT[branch]),
        'face_end': (blocks['face'][1], FACE_END[branch]),
        'presence_mask': (blocks['presence_mask'], MASK_SLICE[branch]),
        'continuous_width': (blocks['presence_mask'][0], CONTINUOUS_WIDTH[branch]),
    }
    wrong = {k: v for k, v in checks.items() if v[0] != v[1]}
    if wrong:
        detail = '; '.join(f'{k}: schema {f}, module {e}' for k, (f, e) in wrong.items())
        raise FeatureSpecError(f'{branch} geometry disagrees with the schema. {detail}')

    expected_start = 0
    for name, count in (('face', FACE_LANDMARK_COUNT[branch]),) + NON_FACE_BLOCKS:
        start, end = blocks[name]
        if start != expected_start or end - start != count * 3:
            raise FeatureSpecError(f'{branch}: block {name} is {(start, end)}, expected '
                                   f'{(expected_start, expected_start + count * 3)}')
        expected_start = end
    if expected_start != CONTINUOUS_WIDTH[branch]:
        raise FeatureSpecError(f'{branch}: continuous blocks end at {expected_start}, '
                               f'expected {CONTINUOUS_WIDTH[branch]}')
    return {'blocks': blocks, 'face_indices': list(schema['face_indices'])}


def local_wrist_columns(branch: str) -> Dict[str, List[int]]:
    '''The columns that phase 5 makes exactly zero in every frame.

    local_hand() subtracts the wrist from every hand landmark, so the wrist's own
    local coordinates are identically zero, and undetected frames are zeroed too.
    Predicted here so notebook 9 can CHECK the prediction rather than infer the
    identity of constant columns from their count.
    '''
    _require_branch(branch)
    offset = FACE_END[branch] + POSE_LANDMARKS * 3 + 2 * HAND_LANDMARKS * 3
    result: Dict[str, List[int]] = {}
    for position, name in enumerate(LOCAL_BLOCKS):
        start = offset + position * HAND_LANDMARKS * 3 + HAND_WRIST * 3
        result[name] = [start, start + 1, start + 2]
    return result


def canonical_fps_mapping(fps_by_clip: Dict[str, Any],
                         clip_ids: Sequence[str]) -> str:
    '''The exact clip -> frame-rate mapping, as bytes that can be hashed.

    Each value is written with float.hex(), which is an exact, round-trippable and
    locale-independent representation of the IEEE-754 double. Decimal formatting
    would lose bits and could differ between platforms, so the identity of the
    time metadata would not be reproducible.
    '''
    lines = []
    for clip_id in sorted(str(c) for c in clip_ids):
        if clip_id not in fps_by_clip:
            raise ValueError(f'{clip_id} has no frame rate, cannot canonicalise')
        value = validate_fps(fps_by_clip[clip_id], clip_id)
        lines.append(f'{clip_id}={float(value).hex()}')
    return '\n'.join(lines) + '\n'


def time_metadata_id(fps_by_clip: Dict[str, Any], clip_ids: Sequence[str]) -> str:
    '''Bind the exact frame rates into provenance.

    Frame rate now determines four families of features (velocity, acceleration,
    mean 3D speed, duration), so the mapping itself is part of the experiment and
    a change to any single clip must change this id.
    '''
    return sha16(canonical_fps_mapping(fps_by_clip, clip_ids))


def time_metadata_gate(fps_by_clip: Dict[str, Any], clip_ids: Sequence[str],
                       conflicts: Optional[Sequence[Any]] = None) -> Dict[str, Any]:
    '''Is the time metadata complete and unambiguous for every frozen clip?

    Heterogeneous frame rates are an EXPECTED property of this dataset, not a
    failure, because every temporal feature is converted to physical time using
    the clip's own rate. What must hold is that each frozen clip has exactly one
    finite positive frame rate, with no conflicting value anywhere.
    '''
    ordered = [str(c) for c in clip_ids]
    conflicts = list(conflicts or [])
    missing = [c for c in ordered if c not in fps_by_clip]
    invalid: List[str] = []
    values: List[float] = []
    for clip_id in ordered:
        if clip_id in missing:
            continue
        try:
            values.append(validate_fps(fps_by_clip[clip_id], clip_id))
        except ValueError:
            invalid.append(clip_id)

    array = np.asarray(values, dtype=np.float64)
    result: Dict[str, Any] = {
        'clips_expected': len(ordered),
        'clips_with_one_value': len(ordered) - len(missing) - len(invalid),
        'missing': len(missing), 'missing_examples': missing[:5],
        'invalid': len(invalid), 'invalid_examples': invalid[:5],
        'conflicting': len(conflicts), 'conflicting_examples': conflicts[:5],
        'round_decimals': FPS_ROUND_DECIMALS,
        'rule': 'PASS if every frozen clip has exactly one finite positive frame '
                'rate and no clip has a conflicting value. Heterogeneous frame '
                'rates are expected and are handled by converting temporal '
                'features to physical time, so they are not a failure.',
    }
    if array.size:
        result.update({
            'min': float(array.min()), 'max': float(array.max()),
            'median': float(np.median(array)), 'mean': float(array.mean()),
            'distinct_values': int(np.unique(np.round(array, FPS_ROUND_DECIMALS)).size),
            'nominal_integer_values': sorted({int(round(v)) for v in array}),
            'relative_spread': float((array.max() - array.min()) / np.median(array))
            if float(np.median(array)) > 0 else float('inf'),
        })

    reasons: List[str] = []
    if missing:
        reasons.append(f'{len(missing)} frozen clips have no frame-rate row')
    if invalid:
        reasons.append(f'{len(invalid)} clips have a non-finite or non-positive '
                       'frame rate')
    if conflicts:
        reasons.append(f'{len(conflicts)} clips carry conflicting frame rates')
    result['verdict'] = 'STOP' if reasons else 'PASS'
    result['reason'] = ('; '.join(reasons) if reasons
                        else 'every frozen clip has exactly one finite positive frame '
                             'rate, so every temporal feature can be expressed in '
                             'physical time')
    return result


def feature_manifest_id(payload: Dict[str, Any]) -> str:
    return sha16(json.dumps(payload, sort_keys=True, separators=(',', ':'), default=str))


__all__ = [
    'FEATURE_SPEC_VERSION', 'REQUIRED_EXPERIMENT_ID', 'REQUIRED_LOADER_CONTRACT_ID',
    'REQUIRED_STANDARDISER_SET_ID', 'REQUIRED_LOADER_MODULE_SHA256', 'BRANCHES',
    'BRANCH_WIDTH', 'CONTINUOUS_WIDTH', 'MASK_SLICE', 'FACE_LANDMARK_COUNT',
    'LANDMARK_COUNT', 'FACE_END', 'AXES', 'PER_COORDINATE_STATISTICS',
    'TRACKING_PARTS', 'TRACKING_STATISTICS', 'MIN_FRAMES', 'LOCAL_BLOCKS',
    'HAND_WRIST', 'FeatureSpecError', 'LeakageError', 'ScopeError',
    'feature_count', 'per_coordinate_count', 'block_offsets', 'landmark_layout',
    'build_feature_spec', 'spec_hash', 'cross_branch_index_map',
    'missing_gap_stats', 'extract_clip_features', 'reference_clip_features',
    'extract_matrix', 'FeatureFilter', 'column_extremes',
    'fit_zero_variance_filter', 'require_filter_scope', 'save_filters',
    'load_filters', 'verify_notebook8', 'verify_branch_geometry',
    'local_wrist_columns', 'feature_manifest_id', 'sha16', 'array_digest',
    'file_sha256', 'time_metadata_gate', 'validate_fps',
    'canonical_fps_mapping', 'time_metadata_id', 'FPS_ROUND_DECIMALS',
    'MIN_FPS', 'POSITION_STATISTICS', 'RATE_STATISTICS',
    'VELOCITY_FPS_POWER', 'ACCELERATION_FPS_POWER', 'SPEED_FPS_POWER',
]
"""[1:]


RF_MODULE_PATH.write_text(RF_MODULE_SOURCE, encoding="utf-8")
compile(RF_MODULE_PATH.read_text(encoding="utf-8"), str(RF_MODULE_PATH), "exec")

if str(WORKING) not in sys.path:
    sys.path.insert(0, str(WORKING))
if "bsl_rf_features" in sys.modules:
    del sys.modules["bsl_rf_features"]
import bsl_rf_features as R
importlib.reload(R)

print(f"wrote {RF_MODULE_PATH.name}  {RF_MODULE_PATH.stat().st_size:,} bytes, "
      f"{len(RF_MODULE_SOURCE.splitlines()):,} lines")
print(f"imported from {R.__file__}")

# ---- the module carries the notebook 8 ids; they must agree with cell 1 ----
print("\nCONTRACT CONSTANTS: notebook vs imported module")
for name, found, wanted in (
        ("feature spec version", R.FEATURE_SPEC_VERSION, FEATURE_SPEC_VERSION),
        ("experiment id", R.REQUIRED_EXPERIMENT_ID, REQUIRED_EXPERIMENT_ID),
        ("loader contract id", R.REQUIRED_LOADER_CONTRACT_ID, REQUIRED_LOADER_CONTRACT_ID),
        ("standardiser set id", R.REQUIRED_STANDARDISER_SET_ID, REQUIRED_STANDARDISER_SET_ID),
        ("loader module sha256", R.REQUIRED_LOADER_MODULE_SHA256, REQUIRED_LOADER_MODULE_SHA256),
        ("branch widths", R.BRANCH_WIDTH, EXPECTED["widths"]),
        ("continuous widths", R.CONTINUOUS_WIDTH, EXPECTED["continuous"]),
        ("landmark counts", R.LANDMARK_COUNT, EXPECTED["landmarks"]),
        ("face landmark counts", R.FACE_LANDMARK_COUNT, EXPECTED["face_landmarks"]),
        ("per-coordinate statistics", len(R.PER_COORDINATE_STATISTICS),
         EXPECTED["per_coordinate_statistics"]),
        ("tracking features", len(R.TRACKING_PARTS) * len(R.TRACKING_STATISTICS),
         EXPECTED["tracking_features"]),
        ("hand wrist landmark", R.HAND_WRIST, 0)):
    require(f"module constant: {name}", found == wanted, f"{found!r}")

require("the module verifies notebook 8 independently",
        R.verify_notebook8(LOADER_MANIFEST, LOADER_MODULE_PATH)["module_sha256_on_disk"]
        == REQUIRED_LOADER_MODULE_SHA256, "all four ids")
require("presence-mask columns are excluded from the per-coordinate block",
        all(R.CONTINUOUS_WIDTH[b] == R.MASK_SLICE[b][0] for b in R.BRANCHES),
        "generic statistics stop where the masks start")
require("the eight position statistics receive no explicit FPS scaling",
        R.POSITION_STATISTICS == ("mean", "std", "median", "min", "max", "range",
                                  "rms", "displacement"),
        f"{len(R.POSITION_STATISTICS)} left as sampled-trajectory summaries")
require("the six rate statistics are the ones expressed in physical time",
        R.RATE_STATISTICS == ("velocity_mean_abs", "velocity_std", "velocity_max_abs",
                              "acceleration_mean_abs", "acceleration_std",
                              "acceleration_max_abs"),
        f"{len(R.RATE_STATISTICS)} scaled by f or f^2")
require("frame rate is never itself a feature",
        not any("fps" in str(name).lower() or "frame_rate" in str(name).lower()
                for name in (R.PER_COORDINATE_STATISTICS + R.TRACKING_STATISTICS
                             + ("mean_3d_speed", "duration_seconds"))),
        "fps is auxiliary metadata only")

# ---- geometry checked against the phase 5 schemas on disk ------------------
print("\nBRANCH GEOMETRY (module assumptions vs the phase 5 schema)")
FACE_INDICES: Dict[str, List[int]] = {}
for branch in R.BRANCHES:
    geometry = R.verify_branch_geometry(branch, context.schemas[branch])
    FACE_INDICES[branch] = [int(i) for i in geometry["face_indices"]]
    print(f"  {branch:<8} width={R.BRANCH_WIDTH[branch]} "
          f"continuous={R.CONTINUOUS_WIDTH[branch]} landmarks={R.LANDMARK_COUNT[branch]} "
          f"face_indices={len(FACE_INDICES[branch])}")
    require(f"{branch} geometry agrees with the schema",
            len(FACE_INDICES[branch]) == EXPECTED["face_landmarks"][branch],
            f"{len(FACE_INDICES[branch])} face landmarks")
require("the reduced face landmarks are a subset of the full face landmarks",
        set(FACE_INDICES["reduced"]).issubset(set(FACE_INDICES["full"])),
        f"{len(set(FACE_INDICES['reduced']) & set(FACE_INDICES['full']))} of "
        f"{len(FACE_INDICES['reduced'])}")
require("the full branch keeps every MediaPipe face landmark",
        FACE_INDICES["full"] == list(range(EXPECTED["face_landmarks"]["full"])),
        "0..467")

# ---- the feature specification --------------------------------------------
print("\nFEATURE SPECIFICATION")
SPEC: Dict[str, pd.DataFrame] = {}
SPEC_HASH: Dict[str, str] = {}
for branch in R.BRANCHES:
    SPEC[branch] = R.build_feature_spec(branch, FACE_INDICES[branch])
    SPEC_HASH[branch] = R.spec_hash(SPEC[branch])
    counts = SPEC[branch].groupby("family").size().to_dict()
    print(f"  {branch:<8} features={len(SPEC[branch]):,}  hash={SPEC_HASH[branch]}")
    print(f"           families: {counts}")
    require(f"{branch} feature count matches the published dimension",
            len(SPEC[branch]) == EXPECTED["features"][branch], f"{len(SPEC[branch]):,}")
    require(f"{branch} feature indices are contiguous 0..F-1",
            SPEC[branch]["feature_index"].tolist() == list(range(len(SPEC[branch]))),
            f"0..{len(SPEC[branch]) - 1:,}")
    require(f"{branch} feature names are unique",
            SPEC[branch]["feature_name"].nunique() == len(SPEC[branch]),
            f"{SPEC[branch]['feature_name'].nunique():,}")
    require(f"{branch} shared keys are unique",
            SPEC[branch]["shared_key"].nunique() == len(SPEC[branch]), "unique")
    require(f"{branch} every feature is marked per-clip deterministic",
            set(SPEC[branch]["kind"].unique()) == {"per_clip_deterministic"},
            "the only fold-fitted object is the zero-variance mask")
    require(f"{branch} family sizes match the specification",
            counts == {"global": 1, "mean_3d_speed": EXPECTED["landmarks"][branch],
                       "per_coordinate": 14 * EXPECTED["continuous"][branch],
                       "tracking": 12}, "as published")

print(f"\n  modality breakdown (full):")
print(SPEC["full"].groupby(["family", "modality"]).size().to_string())

# ---- cross-branch correspondence ------------------------------------------
REDUCED_TO_FULL, FULL_FROM_REDUCED = R.cross_branch_index_map(SPEC["full"], SPEC["reduced"])
print(f"\n  cross-branch correspondences: {len(REDUCED_TO_FULL):,}")
require("every reduced feature has a full counterpart",
        len(REDUCED_TO_FULL) == EXPECTED["features"]["reduced"],
        f"{len(REDUCED_TO_FULL):,} of {EXPECTED['features']['reduced']:,}")
require("the correspondence targets distinct full features",
        len(set(FULL_FROM_REDUCED.tolist())) == len(FULL_FROM_REDUCED), "injective")

for branch in R.BRANCHES:
    SPEC[branch].to_csv(SPEC_PATH[branch], index=False)
    print(f"  wrote {SPEC_PATH[branch].name}  {SPEC_PATH[branch].stat().st_size:,} bytes")

In [ ]:
# ============================================================
# CELL 4 — Which base columns are constant, and what are they?
# ============================================================
# The count of constant columns is not evidence of their identity. This cell
# finds them from the notebook 6 sufficient statistics, maps them through the
# feature specification, prints what they are, and only then compares the answer
# against the prediction that they are the two hand-local wrist rows.
#
# Phase 5's local_hand() subtracts the wrist from every hand landmark, so the
# wrist's own local coordinates are identically zero, and undetected frames are
# zeroed as well. That is the prediction under test.

print("PREDICTED CONSTANT COLUMNS (from the phase 5 construction)")
PREDICTED: Dict[str, List[int]] = {}
for branch in R.BRANCHES:
    predicted = R.local_wrist_columns(branch)
    PREDICTED[branch] = sorted(c for columns in predicted.values() for c in columns)
    print(f"  {branch:<8} {predicted}")

print("\nOBSERVED CONSTANT COLUMNS (from the notebook 6 per-signer statistics)")
OBSERVED: Dict[str, List[int]] = {}
for branch in R.BRANCHES:
    statistics = context.signer_statistics(branch)
    total_frames = 0
    total_sum = np.zeros(R.BRANCH_WIDTH[branch], dtype=np.float64)
    total_sumsq = np.zeros(R.BRANCH_WIDTH[branch], dtype=np.float64)
    for signer in sorted(statistics.signers, key=lambda s: int(str(s).lstrip("p"))):
        total_frames += statistics.frames[signer]
        total_sum += statistics.sums[signer]
        total_sumsq += statistics.sumsq[signer]
    require(f"{branch} global statistics cover every frame",
            total_frames == EXPECTED["total_frames"], f"{total_frames:,}")

    continuous = slice(0, R.CONTINUOUS_WIDTH[branch])
    exactly_zero = np.flatnonzero((total_sum[continuous] == 0.0)
                                  & (total_sumsq[continuous] == 0.0))
    mean = total_sum / total_frames
    variance = np.maximum(total_sumsq / total_frames - mean ** 2, 0.0)
    near_zero = np.flatnonzero(variance[continuous] < 1e-12)
    OBSERVED[branch] = sorted(int(c) for c in exactly_zero)

    print(f"  {branch:<8} exactly zero: {OBSERVED[branch]}")
    print(f"           variance below 1e-12: {sorted(int(c) for c in near_zero)}")
    require(f"{branch} has exactly {EXPECTED['constant_base_columns']} constant "
            "continuous columns",
            len(OBSERVED[branch]) == EXPECTED["constant_base_columns"],
            len(OBSERVED[branch]))
    require(f"{branch}: no other continuous column is near-constant",
            sorted(int(c) for c in near_zero) == OBSERVED[branch],
            "exactly-zero and near-zero sets agree")

print("\nIDENTITY OF THE CONSTANT COLUMNS (read from the feature specification)")
CONSTANT_REPORT: List[Dict[str, Any]] = []
for branch in R.BRANCHES:
    rows = SPEC[branch][(SPEC[branch]["statistic"] == "mean")
                        & SPEC[branch]["base_column"].isin(OBSERVED[branch])]
    for row in rows.sort_values("base_column").itertuples():
        CONSTANT_REPORT.append({"branch": branch, "base_column": int(row.base_column),
                                "block": row.block, "modality": row.modality,
                                "source_landmark": int(row.source_landmark),
                                "axis": row.axis})
        print(f"  {branch:<8} column {row.base_column:>5}  block={row.block:<17} "
              f"landmark={row.source_landmark:<3} axis={row.axis}")

    blocks = sorted(set(rows["block"]))
    landmarks = sorted(set(int(v) for v in rows["source_landmark"]))
    axes = sorted(set(rows["axis"]))
    require(f"{branch} constant columns sit only in the hand-local blocks",
            blocks == sorted(R.LOCAL_BLOCKS), blocks)
    require(f"{branch} constant columns are landmark {R.HAND_WRIST} only",
            landmarks == [R.HAND_WRIST], landmarks)
    require(f"{branch} constant columns cover all three axes",
            axes == ["x", "y", "z"], axes)
    require(f"{branch} observed constant columns match the prediction",
            OBSERVED[branch] == PREDICTED[branch],
            f"{OBSERVED[branch]} == {PREDICTED[branch]}")

print("\nVerified: the six constant continuous base columns are exactly")
print("  left_hand_local  wrist (landmark 0) x, y, z")
print("  right_hand_local wrist (landmark 0) x, y, z")
print("in both branches. This is the wrist-origin construction, not a data fault.")

# ---- confirm it at the file level too, not only in the aggregates ----------
sample = (sorted(CLIP_IDS)[:2]
          + [str(FRAMES.idxmin()), str(FRAMES.idxmax())]
          + context.manifest["clip_id"].astype(str)
            .sample(n=min(8, len(context.manifest)), random_state=RANDOM_SEED).tolist())
sample = sorted(dict.fromkeys(sample))
file_level_clean = True
for clip_id in sample:
    for branch in R.BRANCHES:
        array = context.load_raw_sequence(clip_id, branch)
        if np.abs(array[:, OBSERVED[branch]]).max() != 0.0:
            file_level_clean = False
require("the constant columns are exactly 0.0 in every frame of sampled clips",
        file_level_clean, f"{len(sample)} clips x 2 branches")

# ---- what this means downstream -------------------------------------------
DEAD_FEATURE_INDICES: Dict[str, np.ndarray] = {}
for branch in R.BRANCHES:
    spec = SPEC[branch]
    per_coordinate = spec[(spec["family"] == "per_coordinate")
                          & spec["base_column"].isin(OBSERVED[branch])]
    energy = spec[(spec["family"] == "mean_3d_speed")
                  & spec["block"].isin(R.LOCAL_BLOCKS)
                  & (spec["source_landmark"] == R.HAND_WRIST)]
    DEAD_FEATURE_INDICES[branch] = np.sort(np.concatenate([
        per_coordinate["feature_index"].to_numpy(dtype=np.int64),
        energy["feature_index"].to_numpy(dtype=np.int64)]))
    print(f"\n  {branch:<8} features derived from constant columns: "
          f"{len(DEAD_FEATURE_INDICES[branch])} "
          f"({len(per_coordinate)} per-coordinate + {len(energy)} mean 3D speed)")
    require(f"{branch} dead-feature count is 14 x 6 + 2",
            len(DEAD_FEATURE_INDICES[branch]) == 14 * EXPECTED["constant_base_columns"] + 2,
            len(DEAD_FEATURE_INDICES[branch]))
require("both branches carry the same number of dead features",
        len(DEAD_FEATURE_INDICES["full"]) == len(DEAD_FEATURE_INDICES["reduced"]),
        "identical, so the branch comparison is unaffected")
print("\n  These will be removed by the per-fold zero-variance filter in cell 8.")

In [ ]:
# ============================================================
# CELL 5 — TIME-METADATA gate  (runs BEFORE any feature is extracted)
# ============================================================
# Temporal features are expressed in physical time using each clip's own frame
# rate, so heterogeneous frame rates are an EXPECTED property of this dataset
# rather than a failure. What must hold is that the metadata is complete and
# unambiguous:
#
#   PASS  if every one of the 6,000 frozen clips has exactly one finite positive
#         frame rate, with no conflicting value anywhere.
#   STOP  on any missing, invalid or conflicting frame rate. Nothing is discarded
#         and the gate is never bypassed.
#
# The frame rate is auxiliary metadata used only for the unit conversion. It is
# never a Random Forest predictor.

def _refuses_missing_clip(module, mapping, clip_ids) -> bool:
    """The canonical mapping must refuse to hash an incomplete set."""
    incomplete = {k: v for k, v in mapping.items() if k != clip_ids[0]}
    try:
        module.time_metadata_id(incomplete, clip_ids)
    except ValueError:
        return True
    return False


print("PHASE 3 EXTRACTION METADATA")
fps_sources = sorted(Path(SEARCH_ROOT).glob("**/holistic_metadata_*.csv"))
if not fps_sources:
    raise FileNotFoundError(
        "No Phase 3 extraction metadata (holistic_metadata_*.csv) found under "
        f"{SEARCH_ROOT}. Temporal features cannot be expressed in physical time "
        "without it. Attach the Phase 3 landmark-extraction notebook output as an "
        "additional input and rerun.")
for path in fps_sources:
    print(f"  {path}")

frames_list = [pd.read_csv(path).assign(__source__=path.name) for path in fps_sources]
phase3 = pd.concat(frames_list, ignore_index=True)
print(f"  rows read: {len(phase3):,} from {len(fps_sources)} file(s)")
require("Phase 3 metadata carries an fps column", "fps" in phase3.columns,
        f"{len(phase3.columns)} columns")

if "relative_video_path" in phase3.columns:
    join_field = "relative_video_path"
elif "landmark_file_relative" in phase3.columns:
    join_field = "landmark_file_relative"
else:
    raise RuntimeError("Phase 3 metadata has no path column to derive clip ids from")
phase3["clip_id"] = phase3[join_field].astype(str).map(lambda v: Path(str(v)).stem)
phase3["fps_numeric"] = pd.to_numeric(phase3["fps"], errors="coerce")
print(f"  clip ids derived from {join_field}")

# ---- one unambiguous value per clip ---------------------------------------
conflicting: List[Tuple[str, List[float]]] = []
duplicated = phase3[phase3.duplicated("clip_id", keep=False)]
if not duplicated.empty:
    for clip_id, group in duplicated.groupby("clip_id"):
        values = sorted({round(float(v), R.FPS_ROUND_DECIMALS)
                         for v in group["fps_numeric"] if pd.notna(v)})
        if len(values) != 1:
            conflicting.append((str(clip_id), values))
    print(f"  duplicate clip ids across shards: {duplicated['clip_id'].nunique():,} "
          f"({len(conflicting)} with conflicting or unusable values)")
phase3 = phase3.drop_duplicates("clip_id", keep="first").set_index("clip_id")

FPS_BY_CLIP = {clip_id: (float(phase3.at[clip_id, "fps_numeric"])
                         if clip_id in phase3.index
                         and pd.notna(phase3.at[clip_id, "fps_numeric"]) else None)
               for clip_id in CLIP_IDS}
extra = sorted(set(phase3.index) - set(CLIP_IDS))
GATE = R.time_metadata_gate(FPS_BY_CLIP, CLIP_IDS, conflicting)

print(f"  matched to the frozen manifest: {GATE['clips_with_one_value']:,} of "
      f"{GATE['clips_expected']:,}")
print(f"  Phase 3 rows outside the manifest: {len(extra)}")

# ---- the audit -------------------------------------------------------------
print("\nFPS AUDIT")
print(f"  clips expected       : {GATE['clips_expected']:,}")
print(f"  exactly one value    : {GATE['clips_with_one_value']:,}")
print(f"  missing              : {GATE['missing']:,}  {GATE['missing_examples']}")
print(f"  invalid / zero       : {GATE['invalid']:,}  {GATE['invalid_examples']}")
print(f"  conflicting          : {GATE['conflicting']:,}  "
      f"{[c[0] for c in GATE['conflicting_examples']]}")
if "min" in GATE:
    print(f"  min / median / max   : {GATE['min']:.6f} / {GATE['median']:.6f} / "
          f"{GATE['max']:.6f}")
    print(f"  mean                 : {GATE['mean']:.6f}")
    print(f"  distinct values      : {GATE['distinct_values']:,} "
          f"(variable-frame-rate encoding jitter)")
    print(f"  nominal integer rates: {GATE['nominal_integer_values']}")
    print(f"  relative spread      : {GATE['relative_spread']:.6f}  "
          f"(reported, not gated: temporal features are FPS-normalised)")

usable = pd.Series({c: v for c, v in FPS_BY_CLIP.items() if v is not None},
                   dtype=float).reindex([c for c in CLIP_IDS if FPS_BY_CLIP[c] is not None])
NOMINAL = usable.round().astype("Int64")

print("\n  counts by nominal integer frame rate")
nominal_counts = NOMINAL.value_counts().sort_index()
for value, count in nominal_counts.items():
    print(f"    {int(value):>4} fps  {count:>6,} clips "
          f"({100 * count / len(usable):>5.1f}%)")

# Regime summary: the two recording setups this dataset actually contains.
regime = pd.cut(usable, bins=[0, 40, np.inf], labels=["~30 fps", "~60 fps"])
print("\n  recording regimes")
regime_summary = usable.groupby(regime, observed=False).agg(
    clips="size", minimum="min", maximum="max", mean="mean")
print(regime_summary.to_string())

meta = context.manifest.set_index("clip_id").loc[usable.index, ["signer_id", "session"]]
breakdown = meta.copy()
breakdown["fps"] = usable.to_numpy()
breakdown["nominal"] = NOMINAL.to_numpy()
breakdown["regime"] = regime.to_numpy()

print("\n  by signer")
signer_view = breakdown.groupby("signer_id").agg(
    clips=("fps", "size"), distinct=("fps", "nunique"),
    nominal_rates=("nominal", lambda s: sorted(set(int(v) for v in s))),
    minimum=("fps", "min"), maximum=("fps", "max"),
    regimes=("regime", lambda s: sorted(set(str(v) for v in s))))
signer_view = signer_view.reindex(
    sorted(signer_view.index, key=lambda s: int(str(s).lstrip("p"))))
print(signer_view.to_string())

print("\n  by session")
session_view = breakdown.groupby("session").agg(
    clips=("fps", "size"), distinct=("fps", "nunique"),
    nominal_rates=("nominal", lambda s: sorted(set(int(v) for v in s))),
    minimum=("fps", "min"), maximum=("fps", "max"))
print(session_view.to_string())

print("\n  signer x recording regime")
print(pd.crosstab(breakdown["signer_id"], breakdown["regime"]).reindex(
    sorted(breakdown["signer_id"].unique(),
           key=lambda s: int(str(s).lstrip("p")))).to_string())
print("\n  session x recording regime")
print(pd.crosstab(breakdown["session"], breakdown["regime"]).to_string())

duration = (FRAMES.loc[usable.index].to_numpy(dtype=float) / usable.to_numpy())
print(f"\n  duration in seconds  : {duration.min():.2f}-{duration.max():.2f} "
      f"(mean {duration.mean():.2f})")
print("  Duration is consistent across regimes even though frame counts are not,")
print("  which is exactly why duration_seconds replaces raw frame count as the")
print("  predictor and why velocity, acceleration and speed carry the f factor.")

FPS_AUDIT = {
    "sources": [p.name for p in fps_sources], "join_field": join_field,
    "rows_read": int(len(phase3)),
    "rows_outside_manifest": int(len(extra)),
    "counts_by_nominal_fps": {str(int(v)): int(c) for v, c in nominal_counts.items()},
    "recording_regimes": {str(k): {"clips": int(v["clips"]),
                                   "min": float(v["minimum"]),
                                   "max": float(v["maximum"]),
                                   "mean": float(v["mean"])}
                          for k, v in regime_summary.to_dict("index").items()},
    "by_signer": {str(k): {"clips": int(v["clips"]), "distinct": int(v["distinct"]),
                           "nominal_rates": list(v["nominal_rates"]),
                           "min": float(v["minimum"]), "max": float(v["maximum"]),
                           "regimes": list(v["regimes"])}
                  for k, v in signer_view.to_dict("index").items()},
    "by_session": {str(k): {"clips": int(v["clips"]), "distinct": int(v["distinct"]),
                            "nominal_rates": list(v["nominal_rates"]),
                            "min": float(v["minimum"]), "max": float(v["maximum"])}
                   for k, v in session_view.to_dict("index").items()},
    "duration_seconds": {"min": float(duration.min()), "max": float(duration.max()),
                         "mean": float(duration.mean())},
    "measurement_basis": "OpenCV CAP_PROP_FPS from the Phase 3 extraction metadata, "
                         "treated as the clip-level nominal or average temporal "
                         "sampling rate",
    "limitation": "if source files use variable frame timing internally, exact "
                  "per-frame timestamps are not recoverable from the extracted "
                  "landmark sequence, so the per-second conversion is an "
                  "approximation at that granularity",
    "role": "auxiliary metadata for unit conversion only; the frame rate is never a "
            "Random Forest predictor",
    "decision": GATE,
}
FPS_AUDIT_PATH = WORKING / "rf_fps_audit.json"
FPS_AUDIT_PATH.write_text(json.dumps(FPS_AUDIT, indent=2, sort_keys=True, default=str),
                          encoding="utf-8")
print(f"\nwrote {FPS_AUDIT_PATH.name}")

# ---- the gate --------------------------------------------------------------
print(f"\n  gate rule            : {GATE['rule']}")
print(f"  VERDICT              : {GATE['verdict']}  -  {GATE['reason']}")

if GATE["verdict"] != "PASS":
    record("time-metadata gate", False, GATE["reason"])
    diagnostic = {
        "notebook": "9-random-forest-feature-engineering",
        "verdict": "STOPPED_AT_TIME_METADATA_GATE", "stage": "time_metadata_gate",
        "fps_audit": FPS_AUDIT, "checks": CHECKS,
        "runtime_seconds": round(time.time() - STARTED, 1),
    }
    REPORT_JSON.write_text(json.dumps(diagnostic, indent=2, default=str),
                           encoding="utf-8")
    pd.DataFrame(CHECKS)[["check", "status", "passed", "detail"]].to_csv(
        REPORT_CSV, index=False)
    print("\n" + "=" * 78)
    print("STOPPED AT THE TIME-METADATA GATE - NO FEATURES EXTRACTED")
    print("=" * 78)
    print("  Some clips have a missing, invalid or conflicting frame rate, so their")
    print("  temporal features cannot be expressed in physical time. No clip is")
    print("  discarded and the gate is not bypassed: repair or re-extract the Phase 3")
    print("  metadata for the clips listed above, then rerun.")
    print("=" * 78)
    raise RuntimeError(
        f"Time-metadata gate STOP: {GATE['reason']}. Feature extraction was not "
        "started.")

require("time-metadata gate", True, GATE["reason"])
require("every frozen clip has exactly one frame-rate value",
        GATE["clips_with_one_value"] == EXPECTED["clips"],
        f"{GATE['clips_with_one_value']:,} of {EXPECTED['clips']:,}")
require("no frozen clip has a missing frame rate", GATE["missing"] == 0, GATE["missing"])
require("no frozen clip has an invalid or non-positive frame rate",
        GATE["invalid"] == 0, GATE["invalid"])
require("no frozen clip has a conflicting frame rate", GATE["conflicting"] == 0,
        GATE["conflicting"])
require("every frame rate passes the extractor's own validation",
        all(R.validate_fps(FPS_BY_CLIP[c], c) > 0 for c in CLIP_IDS),
        f"{EXPECTED['clips']:,} of {EXPECTED['clips']:,}")

FPS = pd.Series({c: float(FPS_BY_CLIP[c]) for c in CLIP_IDS}, dtype=float).loc[CLIP_IDS]

# ---- bind the exact frame rates into provenance ----------------------------
# Frame rate now determines four families of features, so the mapping itself is
# part of the experiment. It is hashed from float.hex() values, which are exact
# and locale independent, and persisted so notebook 10 can re-verify it.
print("\nTIME-METADATA PROVENANCE")
SOURCE_SHA256 = {path.name: R.file_sha256(path) for path in fps_sources}
for name, digest in SOURCE_SHA256.items():
    print(f"  {name}")
    print(f"    sha256 {digest}")
TIME_METADATA_ID = R.time_metadata_id(FPS_BY_CLIP, CLIP_IDS)
print(f"  clips bound          : {len(CLIP_IDS):,}")
print(f"  representation       : float.hex() per clip, sorted by clip id")
print(f"  time metadata id     : {TIME_METADATA_ID}")

# fps_hex is the AUTHORITATIVE column: it is exact and locale independent, and it
# is what time_metadata_id is computed from. The decimal fps column is a
# human-readable convenience. It is written with 17 significant digits, which is
# enough to round-trip any float64, because some pandas versions otherwise emit
# 16 and lose the last bit.
FPS_TABLE_PATH = WORKING / "rf_fps_by_clip.csv"
pd.DataFrame({"clip_id": CLIP_IDS,
              "fps": [float(FPS_BY_CLIP[c]) for c in CLIP_IDS],
              "fps_hex": [float(FPS_BY_CLIP[c]).hex() for c in CLIP_IDS]}).to_csv(
    FPS_TABLE_PATH, index=False, float_format="%.17g")
print(f"  wrote {FPS_TABLE_PATH.name}  {FPS_TABLE_PATH.stat().st_size:,} bytes")

# Round trip through the persisted file, not through memory.
persisted = pd.read_csv(FPS_TABLE_PATH, dtype={"clip_id": str, "fps_hex": str})
rebuilt = {row.clip_id: float.fromhex(row.fps_hex) for row in persisted.itertuples()}
require("the persisted frame rates reproduce the time metadata id",
        R.time_metadata_id(rebuilt, CLIP_IDS) == TIME_METADATA_ID, TIME_METADATA_ID)
require("the persisted hex values round-trip to the exact frame rates",
        all(rebuilt[c] == float(FPS_BY_CLIP[c]) for c in CLIP_IDS),
        f"{len(CLIP_IDS):,} clips, bit-exact")
# Reported against a tolerance, not bit-equality: the decimal column is a
# convenience and its precision depends on the pandas writer, whereas every value
# used by the pipeline comes from fps_hex.
decimal_column = persisted["fps"].to_numpy(dtype=np.float64)
authoritative = np.array([rebuilt[c] for c in persisted["clip_id"]], dtype=np.float64)
decimal_drift = float(np.abs(decimal_column - authoritative).max())
relative_drift = float(np.abs((decimal_column - authoritative) / authoritative).max())
print(f"  decimal column drift : {decimal_drift:.3e} absolute, "
      f"{relative_drift:.3e} relative (hex column is authoritative)")
require("the convenience decimal column agrees with the authoritative hex column",
        relative_drift <= 1e-12,
        f"relative {relative_drift:.3e} <= 1e-12"
        + (" (bit-exact)" if decimal_drift == 0.0 else ""))

# Changing one clip's frame rate must change the id.
mutated = dict(FPS_BY_CLIP)
victim = CLIP_IDS[0]
mutated[victim] = float(np.nextafter(float(FPS_BY_CLIP[victim]), np.inf))
require("a one-ulp change to a single clip's frame rate changes the id",
        R.time_metadata_id(mutated, CLIP_IDS) != TIME_METADATA_ID,
        f"{victim} perturbed -> {R.time_metadata_id(mutated, CLIP_IDS)}")
doubled = dict(FPS_BY_CLIP)
doubled[victim] = float(FPS_BY_CLIP[victim]) * 2.0
require("doubling one clip's frame rate changes the id",
        R.time_metadata_id(doubled, CLIP_IDS) != TIME_METADATA_ID,
        R.time_metadata_id(doubled, CLIP_IDS))
require("the id is order independent",
        R.time_metadata_id(FPS_BY_CLIP, list(reversed(CLIP_IDS))) == TIME_METADATA_ID,
        "canonicalised by sorted clip id")
require("a clip missing from the mapping is refused",
        _refuses_missing_clip(R, FPS_BY_CLIP, CLIP_IDS), "ValueError")

TIME_METADATA_PROVENANCE = {
    "time_metadata_id": TIME_METADATA_ID,
    "clip_count": len(CLIP_IDS),
    "representation": "sha256 over sorted lines 'clip_id=float.hex(fps)', truncated "
                      "to 16 hex characters; float.hex is exact and locale "
                      "independent, decimal formatting is not",
    "source_files": SOURCE_SHA256,
    "source_description": "Phase 3 extraction metadata, cv2.CAP_PROP_FPS per video",
    "persisted_as": FPS_TABLE_PATH.name,
    "validation_rule": GATE["rule"],
    "verdict": GATE["verdict"],
}
FPS_AUDIT["time_metadata_provenance"] = TIME_METADATA_PROVENANCE
FPS_AUDIT_PATH.write_text(json.dumps(FPS_AUDIT, indent=2, sort_keys=True, default=str),
                          encoding="utf-8")

print("\n  Gate PASSED. Heterogeneous frame rates are expected here and are handled")
print("  by conversion, not by exclusion: velocity carries a factor f, acceleration")
print("  f squared, mean 3D speed f, and duration_seconds is T / f. Raw frame count")
print("  is not a predictor. The frame rate itself is not a predictor. The audit is")
print("  frozen into the feature contract.")

In [ ]:
# ============================================================
# CELL 6 — Extractor self-tests: analytic, FPS scaling, reference path, edge cases
# ============================================================

STAT_POSITION = {statistic: position for position, statistic
                 in enumerate(R.PER_COORDINATE_STATISTICS)}


def coordinate_feature(features: np.ndarray, statistic: str, branch: str,
                       column: int = 0) -> float:
    return float(features[STAT_POSITION[statistic] * R.CONTINUOUS_WIDTH[branch] + column])


def synthetic_clip(frames: int, fps: float, trajectory, branch: str = "reduced",
                   detected: bool = True) -> np.ndarray:
    """Landmark 0's x, y and z all follow the same function of PHYSICAL time."""
    array = np.zeros((frames, R.BRANCH_WIDTH[branch]), dtype=np.float32)
    seconds = np.arange(frames, dtype=np.float64) / fps
    for axis in range(3):
        array[:, axis] = trajectory(seconds).astype(np.float32)
    array[:, R.CONTINUOUS_WIDTH[branch]:] = 1.0 if detected else 0.0
    return array


print("ANALYTIC CHECK (a ramp in frame index at 1 fps, so the closed forms are plain)")
ramp_frames, ramp_branch, ramp_fps = 5, "reduced", 1.0
ramp = synthetic_clip(ramp_frames, ramp_fps, lambda t: t, ramp_branch)
ramp_features = R.extract_clip_features(ramp, ramp_branch, ramp_fps)
offsets = R.block_offsets(ramp_branch)
ANALYTIC = {
    "mean": 2.0, "std": float(np.sqrt(2.0)), "median": 2.0, "min": 0.0, "max": 4.0,
    "range": 4.0, "rms": float(np.sqrt(6.0)), "displacement": 4.0,
    "velocity_mean_abs": 1.0, "velocity_std": 0.0, "velocity_max_abs": 1.0,
    "acceleration_mean_abs": 0.0, "acceleration_std": 0.0, "acceleration_max_abs": 0.0,
}
print(f"  {'statistic':<24}{'computed':>12}{'expected':>12}")
analytic_clean = True
for statistic in R.PER_COORDINATE_STATISTICS:
    computed = coordinate_feature(ramp_features, statistic, ramp_branch)
    expected = ANALYTIC[statistic]
    print(f"  {statistic:<24}{computed:>12.6f}{expected:>12.6f}")
    if abs(computed - expected) > 1e-6:
        analytic_clean = False
require("every per-coordinate statistic matches its closed form", analytic_clean,
        "14 of 14")
require("mean 3D speed on a unit ramp at 1 fps equals sqrt(3)",
        abs(float(ramp_features[offsets["mean_3d_speed"]]) - float(np.sqrt(3.0))) < 1e-6,
        f"{float(ramp_features[offsets['mean_3d_speed']]):.7f}")
require("duration_seconds is T / f",
        float(ramp_features[offsets["duration_seconds"]]) == ramp_frames / ramp_fps,
        ramp_features[offsets["duration_seconds"]])
require("a fully detected clip reports detection rate 1 and no gaps",
        ramp_features[offsets["tracking"]] == 1.0
        and ramp_features[offsets["tracking"] + 1] == 0.0
        and ramp_features[offsets["tracking"] + 2] == 0.0, "1.0 / 0 / 0.0")

# ---- the same physical motion recorded at 30 and at 60 fps -----------------
# This is the check the whole FPS policy exists for. Both clips describe the
# identical continuous trajectory over the identical 2 seconds; only the sampling
# rate differs. Every temporal feature must agree.
print("\nSAME PHYSICAL MOTION AT 30 fps AND 60 fps")
SLOPE, CURVATURE = 1.7, 3.4
linear_30 = synthetic_clip(60, 30.0, lambda t: SLOPE * t)
linear_60 = synthetic_clip(120, 60.0, lambda t: SLOPE * t)
features_30 = R.extract_clip_features(linear_30, "reduced", 30.0)
features_60 = R.extract_clip_features(linear_60, "reduced", 60.0)

print(f"  constant velocity {SLOPE} units/s, 60 frames at 30 fps vs 120 at 60 fps")
print(f"  {'feature':<24}{'30 fps':>12}{'60 fps':>12}{'rel diff':>12}")
linear_clean = True
for statistic, expected in (("velocity_mean_abs", SLOPE), ("velocity_max_abs", SLOPE)):
    low = coordinate_feature(features_30, statistic, "reduced")
    high = coordinate_feature(features_60, statistic, "reduced")
    relative = abs(low - high) / max(abs(low), 1e-12)
    print(f"  {statistic:<24}{low:>12.6f}{high:>12.6f}{relative:>12.2e}")
    if relative > 1e-5 or abs(low - expected) > 1e-4:
        linear_clean = False
speed_30 = float(features_30[offsets["mean_3d_speed"]])
speed_60 = float(features_60[offsets["mean_3d_speed"]])
expected_speed = SLOPE * float(np.sqrt(3.0))
print(f"  {'mean_3d_speed':<24}{speed_30:>12.6f}{speed_60:>12.6f}"
      f"{abs(speed_30 - speed_60) / speed_30:>12.2e}   expected {expected_speed:.6f}")
duration_30 = float(features_30[offsets["duration_seconds"]])
duration_60 = float(features_60[offsets["duration_seconds"]])
print(f"  {'duration_seconds':<24}{duration_30:>12.6f}{duration_60:>12.6f}"
      f"{abs(duration_30 - duration_60):>12.2e}")

require("FPS-corrected velocity agrees across recording rates", linear_clean,
        f"{SLOPE} units/s recovered at both rates")
require("mean 3D speed agrees across recording rates",
        abs(speed_30 - speed_60) / speed_30 < 1e-5
        and abs(speed_30 - expected_speed) < 1e-4,
        f"{speed_30:.6f} vs {speed_60:.6f}")
require("duration_seconds agrees across recording rates",
        abs(duration_30 - duration_60) < 1e-6 and abs(duration_30 - 2.0) < 1e-6,
        f"{duration_30:.6f} s both")

# Constant acceleration. The second difference amplifies float32 quantisation by
# f squared, so the residual is compared relatively rather than absolutely.
quadratic_30 = synthetic_clip(60, 30.0, lambda t: 0.5 * CURVATURE * t ** 2)
quadratic_60 = synthetic_clip(120, 60.0, lambda t: 0.5 * CURVATURE * t ** 2)
accel_30 = R.extract_clip_features(quadratic_30, "reduced", 30.0)
accel_60 = R.extract_clip_features(quadratic_60, "reduced", 60.0)
low = coordinate_feature(accel_30, "acceleration_mean_abs", "reduced")
high = coordinate_feature(accel_60, "acceleration_mean_abs", "reduced")
print(f"\n  constant acceleration {CURVATURE} units/s^2")
print(f"  {'acceleration_mean_abs':<24}{low:>12.5f}{high:>12.5f}"
      f"{abs(low - high) / low:>12.2e}")
require("FPS-corrected acceleration agrees across recording rates",
        abs(low - high) / low < 1e-3 and abs(low - CURVATURE) / CURVATURE < 1e-3,
        f"{CURVATURE} units/s^2 recovered at both rates")

# ---- the scaling law itself, algebraically and bit-exactly ------------------
# Same array, two frame rates: velocity must scale by exactly f, acceleration by
# exactly f squared, duration inversely, and positional statistics not at all.
print("\nTIME-SCALING LAW (one array, f then 2f)")
probe = np.random.default_rng(RANDOM_SEED)
scaling_array = probe.normal(0, 1, (11, R.BRANCH_WIDTH["reduced"])).astype(np.float32)
scaling_array[:, R.CONTINUOUS_WIDTH["reduced"]:] = 1.0
at_f = R.extract_clip_features(scaling_array, "reduced", 25.0)
at_2f = R.extract_clip_features(scaling_array, "reduced", 50.0)
print(f"  {'feature':<26}{'ratio at 2f / f':>18}{'expected':>10}")
scaling_clean = True
scaling_cases = [(statistic, 2.0) for statistic in
                 ("velocity_mean_abs", "velocity_std", "velocity_max_abs")]
scaling_cases += [(statistic, 4.0) for statistic in
                  ("acceleration_mean_abs", "acceleration_std", "acceleration_max_abs")]
scaling_cases += [(statistic, 1.0) for statistic in
                  ("mean", "std", "median", "min", "max", "range", "rms", "displacement")]
for statistic, expected in scaling_cases:
    base = coordinate_feature(at_f, statistic, "reduced")
    doubled = coordinate_feature(at_2f, statistic, "reduced")
    ratio = doubled / base if base != 0 else float("nan")
    print(f"  {statistic:<26}{ratio:>18.9f}{expected:>10.1f}")
    if not np.isfinite(ratio) or abs(ratio - expected) > 1e-6:
        scaling_clean = False
speed_ratio = (float(at_2f[offsets["mean_3d_speed"]])
               / float(at_f[offsets["mean_3d_speed"]]))
duration_ratio = (float(at_2f[offsets["duration_seconds"]])
                  / float(at_f[offsets["duration_seconds"]]))
print(f"  {'mean_3d_speed':<26}{speed_ratio:>18.9f}{2.0:>10.1f}")
print(f"  {'duration_seconds':<26}{duration_ratio:>18.9f}{0.5:>10.1f}")
require("velocity scales by exactly f and acceleration by exactly f squared",
        scaling_clean, "doubling f doubles velocity and quadruples acceleration")
require("mean 3D speed scales by exactly f", abs(speed_ratio - 2.0) < 1e-6,
        f"{speed_ratio:.9f}")
require("duration_seconds scales inversely with f", abs(duration_ratio - 0.5) < 1e-6,
        f"{duration_ratio:.9f}")
require("positional statistics are unaffected by the frame rate",
        all(abs(coordinate_feature(at_2f, statistic, "reduced")
                - coordinate_feature(at_f, statistic, "reduced")) == 0.0
            for statistic in ("mean", "std", "median", "min", "max", "range", "rms",
                              "displacement")), "bit-identical")
require("tracking features are unaffected by the frame rate",
        np.array_equal(at_2f[offsets["tracking"]:offsets["tracking"] + 12],
                       at_f[offsets["tracking"]:offsets["tracking"] + 12]),
        "bit-identical")

print("\nMISSING-GAP COUNTING")
gap_cases = (
    ([1, 1, 0, 0, 1, 0, 1, 1, 1, 1], 2, 2, "two gaps, longest 2"),
    ([1, 1, 1, 1, 1, 1], 0, 0, "nothing missing"),
    ([0, 0, 0, 0, 0, 0], 1, 6, "everything missing"),
    ([0, 1, 1, 1, 1, 0], 2, 1, "gaps at both ends"),
    ([1, 0, 1, 0, 1, 0], 3, 1, "alternating"),
)
gap_clean = True
for mask, gaps, longest, description in gap_cases:
    found = R.missing_gap_stats(np.asarray(mask, dtype=np.float32))
    print(f"  {str(mask):<34} -> {found}  expected ({gaps}, {longest})  {description}")
    if found != (gaps, longest):
        gap_clean = False
require("missing-gap counting is correct on every hand-checked pattern", gap_clean,
        f"{len(gap_cases)} of {len(gap_cases)}")

print("\nREFERENCE IMPLEMENTATION (independent loop-based path, both branches)")
reference_clean, worst_reference = True, 0.0
for branch in R.BRANCHES:
    synthetic = probe.normal(0, 1, (9, R.BRANCH_WIDTH[branch])).astype(np.float32)
    synthetic[:, R.CONTINUOUS_WIDTH[branch]:] = (
        probe.random((9, 4)) > 0.35).astype(np.float32)
    for columns in R.local_wrist_columns(branch).values():
        synthetic[:, columns] = 0.0
    for rate in (29.97003, 59.94006):
        fast = R.extract_clip_features(synthetic, branch, rate)
        slow = R.reference_clip_features(synthetic, branch, rate)
        difference = float(np.abs(fast.astype(np.float64)
                                 - slow.astype(np.float64)).max())
        worst_reference = max(worst_reference, difference)
        print(f"  {branch:<8} synthetic clip at {rate:>9.5f} fps: "
              f"max |fast - slow| = {difference:.3e}")

real_probe = sorted(dict.fromkeys([str(FRAMES.idxmin()), sorted(CLIP_IDS)[0]]))
for clip_id in real_probe:
    for branch in R.BRANCHES:
        sequence = context.load_raw_sequence(clip_id, branch)
        rate = float(FPS[clip_id])
        fast = R.extract_clip_features(sequence, branch, rate)
        slow = R.reference_clip_features(sequence, branch, rate)
        difference = float(np.abs(fast.astype(np.float64)
                                 - slow.astype(np.float64)).max())
        worst_reference = max(worst_reference, difference)
        print(f"  {branch:<8} {clip_id} (T={sequence.shape[0]}, "
              f"{rate:.4f} fps): max |fast - slow| = {difference:.3e}")
        if difference > 1e-4:
            reference_clean = False
require("the vectorised extractor matches the independent reference path",
        reference_clean and worst_reference <= 1e-4, f"max {worst_reference:.3e}")

print("\nEDGE CASES AND NEGATIVE TESTS")


def refuses(name: str, call, expected) -> None:
    try:
        call()
    except expected as error:
        record(name, True, type(error).__name__)
        return
    except Exception as error:
        record(name, False, f"raised {type(error).__name__} instead")
        raise RuntimeError(f"{name}: wrong exception type") from error
    record(name, False, "no error raised")
    raise RuntimeError(f"{name}: invalid input was accepted")


# A bad frame rate must stop the clip, never silently fall back to per-frame units.
for bad, label in ((0.0, "zero"), (-30.0, "negative"), (float("nan"), "NaN"),
                   (float("inf"), "infinite"), (None, "missing"),
                   ("thirty", "non-numeric")):
    refuses(f"a {label} frame rate is refused",
            lambda bad=bad: R.extract_clip_features(scaling_array, "reduced", bad),
            (ValueError, TypeError))
refuses("a clip with no frame rate is refused by the matrix builder",
        lambda: R.extract_matrix(lambda c: scaling_array, ["absent"], "reduced", {}),
        ValueError)
refuses("a wrong-width sequence is refused",
        lambda: R.extract_clip_features(np.zeros((10, 999), np.float32), "full", 30.0),
        ValueError)
refuses("a sequence too short for acceleration is refused",
        lambda: R.extract_clip_features(
            np.zeros((2, R.BRANCH_WIDTH["reduced"]), np.float32), "reduced", 30.0),
        ValueError)
refuses("an unknown branch is refused",
        lambda: R.extract_clip_features(np.zeros((10, 631), np.float32), "holistic",
                                        30.0), KeyError)
refuses("a non-finite sequence is refused",
        lambda: R.extract_clip_features(
            np.full((5, R.BRANCH_WIDTH["reduced"]), np.nan, np.float32), "reduced",
            30.0), ValueError)
refuses("a face_indices list of the wrong length is refused",
        lambda: R.build_feature_spec("reduced", list(range(50))), R.FeatureSpecError)
refuses("unsorted face_indices are refused",
        lambda: R.build_feature_spec("reduced", list(reversed(FACE_INDICES["reduced"]))),
        R.FeatureSpecError)

minimum_length = R.extract_clip_features(
    np.tile(np.arange(R.MIN_FRAMES, dtype=np.float32)[:, None],
            (1, R.BRANCH_WIDTH["reduced"])), "reduced", 30.0)
require(f"the shortest permitted clip (T={R.MIN_FRAMES}) extracts cleanly",
        minimum_length.shape == (EXPECTED["features"]["reduced"],)
        and bool(np.isfinite(minimum_length).all()), minimum_length.shape)
require("features are stored as float32",
        minimum_length.dtype == np.float32 and ramp_features.dtype == np.float32,
        "float32")
require("extraction is deterministic",
        R.extract_clip_features(ramp, ramp_branch, ramp_fps).tobytes()
        == ramp_features.tobytes(), R.array_digest(ramp_features))
require("the frame rate is never emitted as a feature",
        not any("fps" in str(value).lower() for branch in R.BRANCHES
                for column in ("family", "statistic", "block", "feature_name")
                for value in SPEC[branch][column].unique()),
        "fps is auxiliary metadata only")

In [ ]:
# ============================================================
# CELL 6 — Extract the 6,000-row feature matrix, one branch at a time
# ============================================================
# Per-clip extraction reads only that clip, so building all 6,000 rows in one
# pass is leakage-free. Every row is written into its own preallocated slot, so
# the result does not depend on worker count or completion order.
#
# Written uncompressed so notebook 10 can memory-map it and slice a fold's rows
# without loading the whole thing.

MATRIX_DIGEST: Dict[str, str] = {}
MATRIX_SHAPE: Dict[str, Tuple[int, int]] = {}
pd.DataFrame({"row": range(len(CLIP_IDS)), "clip_id": CLIP_IDS}).to_csv(
    CLIP_ORDER_PATH, index=False)
print(f"wrote {CLIP_ORDER_PATH.name}  {len(CLIP_IDS):,} clips in sorted order\n")

for branch in R.BRANCHES:
    started = time.time()
    expected_bytes = len(CLIP_IDS) * EXPECTED["features"][branch] * 4
    print(f"{branch.upper()} BRANCH  {len(CLIP_IDS):,} clips x "
          f"{EXPECTED['features'][branch]:,} features "
          f"({expected_bytes / 1e6:.0f} MB)")

    def progress(done: int, total: int, started=started) -> None:
        elapsed = time.time() - started
        rate = done / max(elapsed, 1e-6)
        print(f"    [{done:,}/{total:,}] {elapsed / 60:.1f} min | {rate:.1f} clips/s | "
              f"eta {(total - done) / max(rate, 1e-6) / 60:.1f} min", flush=True)

    matrix, ordered = R.extract_matrix(
        lambda clip_id, branch=branch: context.load_raw_sequence(clip_id, branch),
        CLIP_IDS, branch, FPS_BY_CLIP, workers=EXTRACT_WORKERS, progress=progress)

    require(f"{branch} matrix rows follow the frozen clip order",
            ordered == CLIP_IDS, f"{len(ordered):,} clips")
    require(f"{branch} matrix has the published shape",
            matrix.shape == (len(CLIP_IDS), EXPECTED["features"][branch]),
            str(matrix.shape))
    require(f"{branch} matrix is float32", matrix.dtype == np.float32, matrix.dtype)
    require(f"{branch} matrix is entirely finite", bool(np.isfinite(matrix).all()),
            "no NaN or Inf")

    # duration_seconds = T / f, so it doubles as proof that the extractor saw every
    # original frame: no reduction, no padding, no truncation. Recomputed here from
    # the frozen frame counts and the audited frame rates, then inverted back to a
    # frame count so the original claim is still checked exactly.
    duration_column = matrix[:, R.block_offsets(branch)["duration_seconds"]].astype(np.float64)
    rates = FPS.loc[CLIP_IDS].to_numpy(dtype=np.float64)
    expected_duration = FRAMES.loc[CLIP_IDS].to_numpy(dtype=np.float64) / rates
    worst_duration = float(np.abs(duration_column - expected_duration).max())
    require(f"{branch} duration_seconds equals T / fps for every clip",
            worst_duration < 1e-4, f"max diff {worst_duration:.2e}")
    require(f"{branch} every clip contributed all of its original frames",
            np.array_equal(np.rint(duration_column * rates),
                           FRAMES.loc[CLIP_IDS].to_numpy(dtype=np.float64)),
            f"{int(FRAMES.loc[CLIP_IDS].sum()):,} frames total")
    require(f"{branch} raw frame count is not stored as a feature",
            "sequence_length" not in set(SPEC[branch]["statistic"]),
            "duration_seconds replaces it")

    MATRIX_DIGEST[branch] = R.array_digest(matrix)
    MATRIX_SHAPE[branch] = tuple(int(v) for v in matrix.shape)
    temporary = MATRIX_PATH[branch].parent / (MATRIX_PATH[branch].name + ".tmp")
    with open(temporary, "wb") as handle:
        np.save(handle, matrix, allow_pickle=False)
    os.replace(temporary, MATRIX_PATH[branch])

    print(f"  extracted in {(time.time() - started) / 60:.1f} min | "
          f"digest {MATRIX_DIGEST[branch]}")
    print(f"  wrote {MATRIX_PATH[branch].name}  "
          f"{MATRIX_PATH[branch].stat().st_size:,} bytes")

    reloaded = np.load(MATRIX_PATH[branch], mmap_mode="r")
    require(f"{branch} matrix round-trips from disk",
            tuple(reloaded.shape) == MATRIX_SHAPE[branch]
            and R.array_digest(np.asarray(reloaded)) == MATRIX_DIGEST[branch],
            MATRIX_DIGEST[branch])
    del reloaded, matrix
    gc.collect()
    print()

print("FEATURE VALUE RANGES (sanity, not a gate)")
for branch in R.BRANCHES:
    matrix = np.load(MATRIX_PATH[branch], mmap_mode="r")
    spec = SPEC[branch]
    for family in ("per_coordinate", "mean_3d_speed", "global", "tracking"):
        columns = spec.loc[spec["family"] == family, "feature_index"].to_numpy()
        block = np.asarray(matrix[:, columns[:min(len(columns), 4000)]])
        print(f"  {branch:<8} {family:<15} min {block.min():>12.4f} "
              f"max {block.max():>12.4f}")
    del matrix
    gc.collect()

In [ ]:
# ============================================================
# CELL 7 — Cross-branch containment, proved on all 6,000 rows
# ============================================================
# The reduced representation is a strict column subset of the full one: the 92
# retained face landmarks are the same normalised landmarks, and the whole
# non-face tail is byte-identical (notebook 6 proved that, notebook 8 re-proved
# it at the loader layer). So every one of the 9,000 reduced features must equal
# its full counterpart EXACTLY, for every clip.
#
# This is the strongest available test of the indexing scheme. An off-by-one in
# any block offset, a mis-ordered statistic block, a wrong landmark stride or a
# wrong face-index mapping all fail it. It is checked on the whole matrix rather
# than a sample, because it costs little.

print("CROSS-BRANCH CONTAINMENT  reduced[j] == full[map(j)]  for every clip")
full_matrix = np.load(MATRIX_PATH["full"], mmap_mode="r")
reduced_matrix = np.load(MATRIX_PATH["reduced"], mmap_mode="r")
require("both matrices hold the same clips in the same order",
        full_matrix.shape[0] == reduced_matrix.shape[0] == len(CLIP_IDS),
        f"{len(CLIP_IDS):,} rows")

CHUNK = 256
mismatched_rows: List[str] = []
worst_difference = 0.0
compared = 0
for start in range(0, len(CLIP_IDS), CHUNK):
    stop = min(start + CHUNK, len(CLIP_IDS))
    left = np.asarray(reduced_matrix[start:stop])[:, REDUCED_TO_FULL]
    right = np.asarray(full_matrix[start:stop])[:, FULL_FROM_REDUCED]
    compared += left.size
    if not np.array_equal(left, right):
        difference = np.abs(left.astype(np.float64) - right.astype(np.float64))
        worst_difference = max(worst_difference, float(difference.max()))
        for offset in np.flatnonzero(difference.max(axis=1) > 0.0):
            mismatched_rows.append(CLIP_IDS[start + int(offset)])
print(f"  compared {compared:,} feature values across {len(CLIP_IDS):,} clips")
print(f"  mismatching clips: {len(mismatched_rows)}  max |difference| "
      f"{worst_difference:.3e}")
require("every reduced feature equals its full counterpart bit for bit",
        not mismatched_rows and worst_difference == 0.0,
        mismatched_rows[:3] or f"{compared:,} values, max diff 0.00e+00")

# Broken down so the log says which parts of the indexing were exercised.
containment = SPEC["reduced"].copy()
containment["full_index"] = FULL_FROM_REDUCED
print("\n  containment coverage by family and modality")
print(containment.groupby(["family", "modality"]).size().to_string())
face_rows = int((containment["modality"] == "face").sum())
tail_rows = int((containment["modality"] != "face").sum())
require("containment covers the retained face features",
        face_rows == 14 * 92 * 3 + 92, f"{face_rows:,}")
require("containment covers the shared non-face features",
        tail_rows == EXPECTED["features"]["reduced"] - face_rows, f"{tail_rows:,}")

# ---- the dead features must actually be dead ------------------------------
print("\nDEAD FEATURES (derived from the hand-local wrist rows)")
for branch, matrix in (("full", full_matrix), ("reduced", reduced_matrix)):
    columns = DEAD_FEATURE_INDICES[branch]
    block = np.asarray(matrix[:, columns])
    print(f"  {branch:<8} {len(columns)} features | min {block.min():.3e} "
          f"max {block.max():.3e} | distinct values {len(np.unique(block))}")
    require(f"{branch} features from the constant columns are exactly zero everywhere",
            float(np.abs(block).max()) == 0.0, f"max |value| {float(np.abs(block).max()):.1e}")

# ---- detection rates cross-checked against the notebook 7 manifest --------
# An independent chain: notebook 6 read the mask columns, notebook 7 froze the
# per-clip rates, and this notebook re-derived them from the files.
print("\nDETECTION RATES vs THE FROZEN MANIFEST")
manifest_columns = {"face": "face_detection_rate", "pose": "pose_detection_rate",
                    "left_hand": "left_hand_detection_rate",
                    "right_hand": "right_hand_detection_rate"}
available = [p for p, column in manifest_columns.items()
             if column in context.manifest.columns]
if len(available) != len(manifest_columns):
    skip("detection rates cross-check against the manifest",
         f"manifest lacks {sorted(set(manifest_columns) - set(available))}")
else:
    ordered_manifest = context.manifest.set_index("clip_id").loc[CLIP_IDS]
    worst_rate = 0.0
    for branch, matrix in (("full", full_matrix), ("reduced", reduced_matrix)):
        spec = SPEC[branch]
        for part, column in manifest_columns.items():
            index = int(spec.loc[(spec["family"] == "tracking")
                                 & (spec["block"] == f"presence_mask:{part}")
                                 & (spec["statistic"] == "detection_rate"),
                                 "feature_index"].iloc[0])
            extracted = np.asarray(matrix[:, index], dtype=np.float64)
            recorded = ordered_manifest[column].to_numpy(dtype=np.float64)
            difference = float(np.abs(extracted - recorded).max())
            worst_rate = max(worst_rate, difference)
            print(f"  {branch:<8} {part:<11} max |extracted - manifest| "
                  f"{difference:.3e}")
    require("re-derived detection rates match the frozen per-clip rates",
            worst_rate < 1e-6, f"max {worst_rate:.3e}")

del full_matrix, reduced_matrix
gc.collect()

In [ ]:
# ============================================================
# CELL 8 — The only fold-fitted object: zero-variance keep-masks
# ============================================================
# Fitted on each fold's 12 TRAINING signers only, and separately for the
# signer-overlap control on the 4,500 control-training clips only. Constancy is
# tested as max == min, which is equivalent to zero variance for finite floats
# but cannot be spoofed by cancellation.
#
# No scaling, no selection, no PCA. Trees are scale-invariant, so the only
# transformation applied is dropping features that carry no information at all.

ROW_OF = {clip_id: position for position, clip_id in enumerate(CLIP_IDS)}
FOLD_FILTERS: Dict[str, Dict[str, Any]] = {b: {} for b in R.BRANCHES}
CONTROL_FILTERS: Dict[str, Any] = {}
filter_rows: List[Dict[str, Any]] = []

for branch in R.BRANCHES:
    print(f"{branch.upper()} BRANCH")
    matrix = np.load(MATRIX_PATH[branch])          # in RAM: all fits are column reductions
    require(f"{branch} matrix loaded for fitting has the published shape",
            matrix.shape == (len(CLIP_IDS), EXPECTED["features"][branch]),
            str(matrix.shape))

    for fold in range(EXPECTED["folds"]):
        training_signers = context.folds[fold]["train"]
        held_out = set(context.folds[fold]["validation"]) | set(context.folds[fold]["test"])
        training_clips = context.clip_ids(fold, "train")
        rows = [ROW_OF[c] for c in training_clips]

        # Refuse to fit on anything but this fold's training clips.
        signers_present = set(context.manifest.loc[
            context.manifest["clip_id"].isin(training_clips), "signer_id"])
        if len(training_signers) != EXPECTED["train_signers"] \
                or signers_present != set(training_signers) or (signers_present & held_out):
            raise R.LeakageError(
                f"fold {fold}: refusing to fit on {sorted(signers_present)}; expected "
                f"exactly {sorted(training_signers)}")

        fitted = R.fit_zero_variance_filter(
            matrix, rows, branch, scope=f"fold:{fold}", scope_kind="fold", fold=fold,
            fitted_on={"kind": "primary_fold", "fold": int(fold),
                       "training_signers": list(training_signers),
                       "training_clips": len(training_clips),
                       "excluded_validation": list(context.folds[fold]["validation"]),
                       "excluded_test": list(context.folds[fold]["test"]),
                       "rule": "drop features with max == min across the training clips"})
        FOLD_FILTERS[branch][f"fold_{fold}"] = fitted
        filter_rows.append({"branch": branch, "scope": f"fold_{fold}",
                            "rows": fitted.n_rows, "kept": fitted.kept,
                            "dropped": int(fitted.keep.size - fitted.kept),
                            "digest": fitted.digest()})

    control_clips = context.control_clip_ids("train")
    control_rows = [ROW_OF[c] for c in control_clips]
    require(f"{branch} control filter fits on exactly 4,500 clips",
            len(control_rows) == EXPECTED["control_train_clips"], len(control_rows))
    require(f"{branch} control fitting clips exclude the control test and validation clips",
            not (set(control_clips) & set(context.control_clip_ids("test")))
            and not (set(control_clips) & set(context.control_clip_ids("validation"))),
            "disjoint")
    control_filter = R.fit_zero_variance_filter(
        matrix, control_rows, branch, scope="control", scope_kind="control",
        fitted_on={"kind": "signer_overlap_control", "clips": len(control_clips),
                   "rule": "drop features with max == min across the control-training "
                           "clips only, because the control shares signers between "
                           "train and test"})
    CONTROL_FILTERS[branch] = control_filter
    filter_rows.append({"branch": branch, "scope": "control",
                        "rows": control_filter.n_rows, "kept": control_filter.kept,
                        "dropped": int(control_filter.keep.size - control_filter.kept),
                        "digest": control_filter.digest()})
    del matrix
    gc.collect()

filter_table = pd.DataFrame(filter_rows)
print("\nFILTER SUMMARY")
print(filter_table.to_string(index=False))

# ---- the filters must drop the dead features and nothing essential ---------
for branch in R.BRANCHES:
    dead = DEAD_FEATURE_INDICES[branch]
    all_filters = dict(FOLD_FILTERS[branch])
    all_filters["control"] = CONTROL_FILTERS[branch]
    require(f"{branch} every filter drops all {len(dead)} dead features",
            all(not f.keep[dead].any() for f in all_filters.values()),
            f"{len(all_filters)} filters")
    print(f"  {branch:<8} kept {min(f.kept for f in all_filters.values()):,}-"
          f"{max(f.kept for f in all_filters.values()):,} of "
          f"{EXPECTED['features'][branch]:,} features across the "
          f"{len(all_filters)} filters")
    for fold in range(EXPECTED["folds"]):
        fitted = FOLD_FILTERS[branch][f"fold_{fold}"]
        require(f"{branch} fold {fold} filter saw only its 12 training signers",
                set(fitted.fitted_on["training_signers"]) == set(context.folds[fold]["train"])
                and fitted.n_rows == len(context.clip_ids(fold, "train")),
                f"{fitted.n_rows:,} clips")
    require(f"{branch} no filter was fitted on all 6,000 clips",
            all(f.n_rows < len(CLIP_IDS) for f in all_filters.values()),
            f"max {max(f.n_rows for f in all_filters.values()):,} of {len(CLIP_IDS):,}")

# The kept sets must be identical across the shared features of the two branches,
# because those columns hold identical values.
shared_agreement = True
for fold in range(EXPECTED["folds"]):
    full_keep = FOLD_FILTERS["full"][f"fold_{fold}"].keep[FULL_FROM_REDUCED]
    reduced_keep = FOLD_FILTERS["reduced"][f"fold_{fold}"].keep[REDUCED_TO_FULL]
    if not np.array_equal(full_keep, reduced_keep):
        shared_agreement = False
require("the two branches agree on which shared features are constant",
        shared_agreement, "8 of 8 folds")
control_agreement = np.array_equal(
    CONTROL_FILTERS["full"].keep[FULL_FROM_REDUCED],
    CONTROL_FILTERS["reduced"].keep[REDUCED_TO_FULL])
require("the two control filters agree on the shared features", control_agreement,
        "identical")

# ---- the filter must be exactly train_max > train_min ----------------------
# Recomputed by a second, non-chunked code path, and cross-checked against the
# number of distinct values in the fitting rows. A correct zero-variance filter is
# free to drop ANY feature that is constant within its own fitting set, including
# a tracking feature, so nothing here assumes which features survive.
print("\nFILTER EQUIVALENCE TO train_max > train_min")
equivalence_clean = True
dropped_singleton = True
kept_varying = True
extra_dropped: List[Dict[str, Any]] = []

for branch in R.BRANCHES:
    matrix = np.load(MATRIX_PATH[branch])
    spec = SPEC[branch]
    known_dead = set(int(i) for i in DEAD_FEATURE_INDICES[branch])
    scopes = [(f"fold_{fold}", [ROW_OF[c] for c in context.clip_ids(fold, "train")])
              for fold in range(EXPECTED["folds"])]
    scopes.append(("control", [ROW_OF[c] for c in context.control_clip_ids("train")]))

    for scope, rows in scopes:
        fitted = (FOLD_FILTERS[branch][scope] if scope != "control"
                  else CONTROL_FILTERS[branch])
        block = matrix[np.sort(np.asarray(rows, dtype=np.int64))]
        direct_min = block.min(axis=0)
        direct_max = block.max(axis=0)
        if not np.array_equal(fitted.keep, direct_max > direct_min):
            equivalence_clean = False

        dropped = np.flatnonzero(~fitted.keep)
        for column in dropped:
            if np.unique(block[:, column]).size != 1:
                dropped_singleton = False
        sampled_kept = np.flatnonzero(fitted.keep)[::max(1, fitted.kept // 200)]
        for column in sampled_kept:
            if np.unique(block[:, column]).size < 2:
                kept_varying = False

        beyond = sorted(set(int(c) for c in dropped) - known_dead)
        if beyond:
            rows_spec = spec[spec["feature_index"].isin(beyond)]
            for (family, statistic, modality), group in rows_spec.groupby(
                    ["family", "statistic", "modality"]):
                extra_dropped.append({"branch": branch, "scope": scope,
                                      "family": family, "statistic": statistic,
                                      "modality": modality, "features": len(group)})
    del matrix
    gc.collect()

require("every keep-mask equals train_max > train_min exactly", equivalence_clean,
        "recomputed by an independent path for all 18 filters")
require("every dropped feature has exactly one distinct value in its fitting rows",
        dropped_singleton, "genuinely constant")
require("sampled kept features have at least two distinct values", kept_varying,
        "genuinely varying")

print("\nADDITIONAL DROPPED FEATURES BEYOND THE 86 WRIST-DERIVED ONES")
print("  (reported, never gated: a training-only zero-variance filter may drop any")
print("   feature that is constant within its own fitting set, tracking included)")
if extra_dropped:
    extra_table = (pd.DataFrame(extra_dropped)
                   .groupby(["branch", "family", "statistic", "modality"], as_index=False)
                   ["features"].agg(["max", "sum"]))
    extra_table.columns = ["branch", "family", "statistic", "modality",
                           "max_in_one_filter", "total_across_filters"]
    print(extra_table.to_string(index=False))
    per_scope = pd.DataFrame(extra_dropped).groupby(["branch", "scope"])["features"].sum()
    print("\n  totals per filter")
    print(per_scope.to_string())
else:
    extra_table = pd.DataFrame(columns=["branch", "family", "statistic", "modality",
                                        "max_in_one_filter", "total_across_filters"])
    print("  none: every filter dropped exactly the 86 wrist-derived features")
EXTRA_DROPPED = extra_dropped

# ---- scope guards ----------------------------------------------------------
print("\nSCOPE GUARDS (a filter from the wrong scope must be refused)")
scope_probes = [
    ("fold 0 rows cannot use the fold 1 filter",
     lambda: R.require_filter_scope(FOLD_FILTERS["full"]["fold_1"], "full", "fold", 0)),
    ("full rows cannot use the reduced filter",
     lambda: R.require_filter_scope(FOLD_FILTERS["reduced"]["fold_0"], "full", "fold", 0)),
    ("control rows cannot use a fold filter",
     lambda: R.require_filter_scope(FOLD_FILTERS["full"]["fold_0"], "full", "control")),
    ("fold rows cannot use the control filter",
     lambda: R.require_filter_scope(CONTROL_FILTERS["full"], "full", "fold", 0)),
    ("an ad-hoc filter is refused",
     lambda: R.require_filter_scope(
         R.FeatureFilter(branch="full", scope="probe", scope_kind="ad_hoc", fold=None,
                         keep=np.ones(EXPECTED["features"]["full"], bool), n_rows=1),
         "full", "fold", 0)),
    ("a non-filter object is refused",
     lambda: R.require_filter_scope("not a filter", "full", "fold", 0)),
]
for name, call in scope_probes:
    try:
        call()
        require(name, False, "accepted")
    except R.ScopeError as error:
        record(name, True, str(error)[:56])

for name, call, expected_error in (
        ("a fold filter without a fold number is refused",
         lambda: R.FeatureFilter(branch="full", scope="x", scope_kind="fold", fold=None,
                                 keep=np.ones(EXPECTED["features"]["full"], bool),
                                 n_rows=1), ValueError),
        ("a wrong-length keep mask is refused",
         lambda: R.FeatureFilter(branch="full", scope="x", scope_kind="ad_hoc", fold=None,
                                 keep=np.ones(10, bool), n_rows=1), ValueError),
        ("fitting on no rows is refused",
         lambda: R.fit_zero_variance_filter(
             np.zeros((4, EXPECTED["features"]["reduced"]), np.float32), [], "reduced",
             scope="empty"), ValueError)):
    try:
        call()
        require(name, False, "accepted")
    except expected_error as error:
        record(name, True, type(error).__name__)

# ---- persist ---------------------------------------------------------------
for branch in R.BRANCHES:
    path = R.save_filters(FOLD_FILTER_PATH[branch], FOLD_FILTERS[branch],
                          extra={"scope": "primary_folds",
                                 "fitted_on": "the 12 training signers of each fold"})
    reloaded = R.load_filters(path)
    print(f"\nwrote {path.name}  {path.stat().st_size:,} bytes")
    require(f"{branch} fold filters round-trip from disk",
            all(reloaded[k].digest() == FOLD_FILTERS[branch][k].digest()
                for k in FOLD_FILTERS[branch]), f"{len(reloaded)} entries")
    require(f"{branch} fold filters keep their scope across the round trip",
            all(reloaded[k].scope_kind == "fold" and reloaded[k].fold is not None
                for k in reloaded), "scope_kind=fold")

    path = R.save_filters(CONTROL_FILTER_PATH[branch], {"control": CONTROL_FILTERS[branch]},
                          extra={"scope": "signer_overlap_control",
                                 "fitted_on": "the 4,500 control-training clips only"})
    reloaded = R.load_filters(path)
    print(f"wrote {path.name}  {path.stat().st_size:,} bytes")
    require(f"{branch} control filter round-trips from disk",
            reloaded["control"].digest() == CONTROL_FILTERS[branch].digest(),
            reloaded["control"].digest())
    require(f"{branch} control filter keeps the control scope on reload",
            reloaded["control"].scope_kind == "control"
            and reloaded["control"].fold is None, "scope_kind=control, fold=None")

In [ ]:
# ============================================================
# CELL 9 — Compliance, determinism, and the scope-safe read path
# ============================================================

print("RAW-PATH COMPLIANCE")
compliance_sample = sorted(dict.fromkeys(
    [str(FRAMES.idxmin()), str(FRAMES.idxmax())]
    + context.manifest.loc[context.manifest["original_frames"] > 256,
                           "clip_id"].astype(str).tolist()[:4]
    + context.manifest["clip_id"].astype(str)
      .sample(n=min(12, len(context.manifest)), random_state=RANDOM_SEED).tolist()))

no_reduction = True
no_standardisation = True
for clip_id in compliance_sample:
    for branch in R.BRANCHES:
        raw = context.load_raw_sequence(clip_id, branch)
        if int(raw.shape[0]) != int(FRAMES[clip_id]):
            no_reduction = False
        # the loader hands back the file unmodified; nothing here rescales it
        on_disk = np.load(context.path_for(clip_id, branch), allow_pickle=False)
        if np.ascontiguousarray(on_disk, dtype=np.float32).tobytes() != raw.tobytes():
            no_standardisation = False
require("every clip contributed its original frame count, over 256 included",
        no_reduction, f"{len(compliance_sample)} clips x 2 branches")
require("sequences are read unmodified: no neural standardisation applied",
        no_standardisation, "byte-identical to the file on disk")
require("clips longer than 256 frames were included in full",
        int((FRAMES > 256).sum()) == 72
        and all(int(context.load_raw_sequence(c, "reduced").shape[0]) == int(FRAMES[c])
                for c in context.manifest.loc[context.manifest["original_frames"] > 256,
                                              "clip_id"].astype(str).tolist()[:6]),
        f"{int((FRAMES > 256).sum())} long clips, none truncated")

print("\nDETERMINISM")
determinism_clean = True
for clip_id in compliance_sample[:4]:
    for branch in R.BRANCHES:
        sequence = context.load_raw_sequence(clip_id, branch)
        first = R.extract_clip_features(sequence, branch, FPS[clip_id])
        second = R.extract_clip_features(sequence, branch, FPS[clip_id])
        if first.tobytes() != second.tobytes():
            determinism_clean = False
require("repeated extraction of the same clip is bit-identical", determinism_clean,
        f"{len(compliance_sample[:4])} clips x 2 branches")

# Worker count must not change a single stored value.
threaded_clean = True
for branch in R.BRANCHES:
    subset = compliance_sample[:6]
    one_worker, _ = R.extract_matrix(
        lambda c, branch=branch: context.load_raw_sequence(c, branch), subset, branch,
        {c: float(FPS[c]) for c in subset}, workers=1)
    many_workers, _ = R.extract_matrix(
        lambda c, branch=branch: context.load_raw_sequence(c, branch), subset, branch,
        {c: float(FPS[c]) for c in subset}, workers=4)
    if one_worker.tobytes() != many_workers.tobytes():
        threaded_clean = False
    print(f"  {branch:<8} 1 worker vs 4 workers: "
          f"{'identical' if one_worker.tobytes() == many_workers.tobytes() else 'DIFFERENT'}"
          f"  digest {R.array_digest(one_worker)}")
require("worker count does not change any stored feature value", threaded_clean,
        "row slots are preallocated, so order cannot matter")

# Rows read back from the saved matrix must equal a fresh extraction.
print("\nSAVED MATRIX vs FRESH EXTRACTION")
for branch in R.BRANCHES:
    matrix = np.load(MATRIX_PATH[branch], mmap_mode="r")
    worst = 0.0
    for clip_id in compliance_sample[:6]:
        stored = np.asarray(matrix[CLIP_IDS.index(clip_id)])
        fresh = R.extract_clip_features(
            context.load_raw_sequence(clip_id, branch), branch, FPS[clip_id])
        worst = max(worst, float(np.abs(stored - fresh).max()))
    print(f"  {branch:<8} max |stored - fresh| = {worst:.3e}")
    require(f"{branch} stored rows equal a fresh extraction exactly", worst == 0.0,
            f"{len(compliance_sample[:6])} clips")
    del matrix
    gc.collect()

print("\nSCOPE-SAFE READ PATH (the notebook 8 RF generators agree with direct reads)")
generator_clean = True
for branch in R.BRANCHES:
    subset = sorted(context.clip_ids(0, "test"))[:3]
    for item in context.fold_raw_sequences(0, "test", branch, clip_ids=subset):
        direct = context.load_raw_sequence(item["clip_id"], branch)
        if item["sequence"].tobytes() != direct.tobytes() \
                or item["frames"] != int(FRAMES[item["clip_id"]]) \
                or item["label_id"] != context.label_id_for(item["clip_id"]):
            generator_clean = False
    control_subset = sorted(context.control_clip_ids("test"))[:3]
    for item in context.control_raw_sequences(branch=branch, split="test",
                                              clip_ids=control_subset):
        if item["sequence"].tobytes() != context.load_raw_sequence(
                item["clip_id"], branch).tobytes():
            generator_clean = False
require("fold_raw_sequences and control_raw_sequences agree with direct reads",
        generator_clean, "same bytes, same labels, same frame counts")

print("\nFEATURE SPECIFICATION COMPLETENESS")
for branch in R.BRANCHES:
    spec = pd.read_csv(SPEC_PATH[branch])
    require(f"{branch} written spec round-trips with the same hash",
            R.spec_hash(spec) == SPEC_HASH[branch], SPEC_HASH[branch])
    require(f"{branch} written spec has one row per feature",
            len(spec) == EXPECTED["features"][branch]
            and spec["feature_index"].tolist() == list(range(len(spec))),
            f"{len(spec):,} rows")
    require(f"{branch} every spec row records modality, statistic, kind and branch",
            not spec[["modality", "statistic", "kind", "branch"]].isna().any().any()
            and set(spec["branch"].unique()) == {branch}, "complete")
    per_coordinate = spec[spec["family"] == "per_coordinate"]
    require(f"{branch} per-coordinate rows record a source landmark and axis",
            set(per_coordinate["axis"].unique()) == set(R.AXES)
            and (per_coordinate["source_landmark"] >= 0).all()
            and (per_coordinate["base_column"] >= 0).all(), "mapped")
    require(f"{branch} every per-coordinate statistic appears exactly Dc times",
            per_coordinate.groupby("statistic").size().nunique() == 1
            and int(per_coordinate.groupby("statistic").size().iloc[0])
            == EXPECTED["continuous"][branch],
            f"{EXPECTED['continuous'][branch]:,} each")
    require(f"{branch} the tracking block names all four parts and all three statistics",
            sorted(set(spec.loc[spec['family'] == 'tracking', 'block'])) ==
            sorted(f"presence_mask:{p}" for p in R.TRACKING_PARTS)
            and sorted(set(spec.loc[spec['family'] == 'tracking', 'statistic']))
            == sorted(R.TRACKING_STATISTICS), "12 rows")

In [ ]:
# ============================================================
# CELL 10 — Freeze the feature specification, then write the report
# ============================================================
# Same ordering discipline as notebook 8: the stable ids are hashed, the module
# is hashed and re-read from disk, the manifest is written and read back, and the
# validation report is written only after every one of those checks has run.

early_failures = failures()
if early_failures:
    for entry in early_failures:
        print(f"  FAIL  {entry['check']}: {entry['detail']}")

# ---- A. the stable feature contract ---------------------------------------
FEATURE_CONTRACT = {
    "notebook": "9-random-forest-feature-engineering",
    "feature_spec_version": FEATURE_SPEC_VERSION,
    "module_filename": RF_MODULE_PATH.name,
    "upstream": {
        "experiment_id": REQUIRED_EXPERIMENT_ID,
        "loader_contract_id": REQUIRED_LOADER_CONTRACT_ID,
        "standardiser_set_id": REQUIRED_STANDARDISER_SET_ID,
        "loader_module_sha256": REQUIRED_LOADER_MODULE_SHA256,
    },
    "read_path": "notebook 8 raw-sequence API only: original variable-length "
                 "sequences, no 256-frame reduction, no padding, no neural frame "
                 "standardisation",
    "dimensions": {branch: {
        "continuous_columns": EXPECTED["continuous"][branch],
        "landmarks": EXPECTED["landmarks"][branch],
        "face_landmarks": EXPECTED["face_landmarks"][branch],
        "per_coordinate": 14 * EXPECTED["continuous"][branch],
        "mean_3d_speed": EXPECTED["landmarks"][branch],
        "duration_seconds": 1, "tracking": 12,
        "total": EXPECTED["features"][branch],
        "spec_hash": SPEC_HASH[branch],
    } for branch in R.BRANCHES},
    "formulas": {
        "per_coordinate": list(R.PER_COORDINATE_STATISTICS),
        "std_convention": "population standard deviation, ddof = 0, everywhere "
                          "including the velocity and acceleration blocks",
        "median_convention": "even T averages the two middle values",
        "velocity": "v_t = (x[t+1] - x[t]) * f, for t = 0 .. T-2. Units: normalised "
                    "coordinate units per second.",
        "acceleration": "a_t = (x[t+2] - 2*x[t+1] + x[t]) * f^2, for t = 0 .. T-3. "
                        "Units: normalised coordinate units per second^2.",
        "positional_statistics": "No explicit FPS scaling is applied to mean, std, "
                                 "median, min, max, range, rms or displacement. They "
                                 "summarise the sampled normalised-coordinate "
                                 "trajectory and may retain small sampling-density "
                                 "effects.",
        "displacement": "x_{T-1} - x_0, signed",
        "rms": "sqrt( mean( x_t^2 ) )",
        "mean_3d_speed": "per LANDMARK: f * (1/(T-1)) * sum_{t=0}^{T-2} "
                         "|| p_{t+1} - p_t ||_2. Units: normalised body-scale units "
                         "per second. Deliberately not called motion energy: no mass "
                         "and no squared velocity is involved. It is FPS-corrected "
                         "using the clip-level nominal or average CAP_PROP_FPS, so it "
                         "is comparable across the two recording regimes subject to "
                         "the recorded variable-frame-rate limitation, rather than "
                         "perfectly independent of frame rate. Phase 5 scaled every "
                         "coordinate by shoulder width, so it does not scale with body "
                         "size or camera distance.",
        "duration_seconds": "T / f. Raw frame count T is deliberately NOT a "
                            "predictor because it is mechanically frame-rate "
                            "dependent: the same physical clip would carry a "
                            "different value in each recording regime.",
        "tracking": {
            "parts": list(R.TRACKING_PARTS),
            "detection_rate": "mean of the part's binary presence-mask column",
            "missing_gap_count": "number of maximal runs of consecutive zeros",
            "longest_missing_gap_proportion": "longest run of zeros divided by T",
            "note": "the four presence-mask columns are EXCLUDED from the generic "
                    "per-coordinate block and appear only through these features",
        },
        "compute_dtype": "float64 reductions, stored as float32",
    },
    "leakage_boundary": {
        "per_clip_deterministic": "every feature is a function of one clip only, so "
                                  "the 6,000-row matrix is built once per branch with "
                                  "no leakage",
        "fold_fitted": ["zero_variance_keep_mask"],
        "fold_rule": "fitted on the 12 training signers of the fold only; a filter "
                     "from another fold, another branch, the control or an ad-hoc "
                     "probe raises ScopeError",
        "control_rule": "fitted on the 4,500 control-training clips only, because the "
                        "control shares signers between train and test",
        "zero_variance_definition": "drop a feature when max == min across the fitting "
                                    "rows; equivalent to zero variance for finite "
                                    "floats and immune to cancellation",
        "not_applied": ["scaling", "feature selection", "PCA",
                        "any other data-dependent transform"],
        "scaling_note": "trees are scale-invariant, so no scaler is fitted here. If a "
                        "later notebook ever needs one it must be fitted on the 12 "
                        "training signers of each fold, never on all 6,000 clips.",
    },
    "zero_variance_filter": {
        "definition": "drop a feature when train_max == train_min within the fitting "
                      "rows; equivalent to zero variance for finite floats and immune "
                      "to cancellation",
        "verified": "every keep-mask was recomputed by an independent non-chunked path "
                    "and compared for exact equality; every dropped feature was "
                    "confirmed to hold exactly one distinct value in its fitting rows",
        "scope": "any feature that is constant within a fitting set may be dropped, "
                 "including a tracking feature; no feature is assumed to survive and "
                 "no minimum kept count is imposed",
        "additional_dropped_beyond_wrist": EXTRA_DROPPED,
    },
    "time_base": {
        "policy": "every temporal feature is expressed in physical time using the "
                  "clip's own frame rate f. f is auxiliary metadata only and is "
                  "never itself an RF predictor.",
        "scaling": {"velocity": "x-differences * f^1",
                    "acceleration": "second differences * f^2",
                    "mean_3d_speed": "mean 3D step magnitude * f^1",
                    "duration_seconds": "T / f",
                    "position_statistics": "no explicit FPS scaling is applied; they "
                                           "summarise the sampled trajectory and may "
                                           "retain small sampling-density effects"},
        "raw_frame_count_excluded": "sequence_length was removed as a predictor "
                                    "because it is mechanically frame-rate dependent",
        "gate": GATE["rule"],
        "verdict": GATE["verdict"],
        "reason": GATE["reason"],
        "clips_expected": GATE["clips_expected"],
        "clips_with_one_value": GATE["clips_with_one_value"],
        "missing": GATE["missing"], "invalid": GATE["invalid"],
        "conflicting": GATE["conflicting"],
        "observed_min": GATE.get("min"), "observed_median": GATE.get("median"),
        "observed_max": GATE.get("max"),
        "distinct_values": GATE.get("distinct_values"),
        "nominal_integer_values": GATE.get("nominal_integer_values"),
        "relative_spread": GATE.get("relative_spread"),
        "heterogeneity": "expected and handled: the dataset contains two recording "
                         "regimes near 30 and 60 fps, confounded with the signer, and "
                         "physical-time scaling is what makes clips comparable across "
                         "them. Heterogeneous frame rates are not a failure condition.",
        "source": "Phase 3 extraction metadata (cv2.CAP_PROP_FPS per video)",
        "limitation": "CAP_PROP_FPS is treated as the clip-level nominal or average "
                      "temporal sampling rate. If source files use variable frame "
                      "timing internally, exact per-frame timestamps are not "
                      "recoverable from the extracted landmark sequence, so the "
                      "per-second scaling is an approximation at the clip level. This "
                      "is a stated limitation of the dataset and extraction, not "
                      "something notebook 9 can correct.",
    },
    "time_metadata_provenance": TIME_METADATA_PROVENANCE,
    "constant_base_columns": {
        "count": EXPECTED["constant_base_columns"],
        "identity": "left_hand_local and right_hand_local landmark 0 (the wrist), x, y "
                    "and z, in both branches",
        "cause": "phase 5 local_hand() subtracts the wrist from every hand landmark, "
                 "so the wrist's own local coordinates are identically zero",
        "verified": "found from the notebook 6 sufficient statistics, mapped through "
                    "the feature specification, and confirmed at file level; not "
                    "inferred from the count",
        "derived_dead_features": int(len(DEAD_FEATURE_INDICES["full"])),
        "effect": "identical in both branches, so the branch comparison is unaffected; "
                  "removed by the per-fold zero-variance filter",
    },
    "cross_branch_containment": {
        "claim": "every one of the reduced branch's features equals its full-branch "
                 "counterpart exactly, because the reduced representation is a strict "
                 "column subset of the full one",
        "correspondences": int(len(REDUCED_TO_FULL)),
        "verified_on": "all 6,000 clips, not a sample",
    },
    "interpretation_caveats": {
        "dimensionality_confound": "the full branch yields "
            f"{EXPECTED['features']['full']:,} features and the reduced branch "
            f"{EXPECTED['features']['reduced']:,}, a ratio of "
            f"{EXPECTED['features']['full'] / EXPECTED['features']['reduced']:.2f}. This "
            "is a representation-level confound: a Random Forest difference between "
            "branches partly reflects feature dimensionality, not only facial "
            "information. It must be stated wherever RF full-vs-reduced results are "
            "reported.",
        "primary_comparison": "native full representation vs native reduced "
                              "representation. Any matched-dimensionality analysis is "
                              "SECONDARY / SENSITIVITY only, its exact method frozen "
                              "before it is run, and it never replaces the primary "
                              "native comparison.",
        "interpolation": "phase 5 filled undetected frames by linear interpolation, so "
                         "velocity, acceleration and mean-3D-step statistics partly "
                         "reflect interpolation behaviour rather than signing. Applied "
                         "identically to both branches, so it does not bias the branch "
                         "comparison and is not leakage. The tracking features "
                         "quantify it.",
        "no_causal_claim": "a reduced-branch advantage does not show that face "
                          "landmarks are uninformative; dimensionality and RF "
                          "behaviour are confounded with the representation.",
    },
    "downstream_obligations": {
        "notebook_10": "load the matrix with np.load(..., mmap_mode='r'), take the "
                       "fold's rows via the notebook 8 fold membership, apply THAT "
                       "fold's keep-mask through require_filter_scope, and train. "
                       "Freeze the Random Forest hyperparameters before any test "
                       "evaluation and keep them identical between the full and "
                       "reduced branches. Run the signer-overlap control with the "
                       "control filter only.",
        "notebook_13": "report per-modality feature importance using the modality "
                       "column of the feature specification, and carry the "
                       "dimensionality confound and the interpolation caveat into the "
                       "interpretation.",
    },
    "forbidden": ["regenerating folds", "fitting any filter on validation or test "
                  "signers", "applying a filter from another scope",
                  "reducing or padding sequences on the RF path",
                  "replacing the primary native comparison with a "
                  "dimensionality-matched one"],
}
FEATURE_CONTRACT_ID = R.feature_manifest_id(FEATURE_CONTRACT)

FILTER_DIGESTS = {
    "folds": {branch: {key: FOLD_FILTERS[branch][key].digest()
                       for key in sorted(FOLD_FILTERS[branch])} for branch in R.BRANCHES},
    "control": {branch: CONTROL_FILTERS[branch].digest() for branch in R.BRANCHES},
}
FILTER_SET_ID = R.feature_manifest_id(FILTER_DIGESTS)

print("FEATURE CONTRACT")
print(f"  feature contract id  : {FEATURE_CONTRACT_ID}   (formulas, dimensions, rules)")
print(f"  filter set id        : {FILTER_SET_ID}   (the 18 fitted keep-masks)")
for branch in R.BRANCHES:
    print(f"  {branch:<8} spec hash {SPEC_HASH[branch]}  matrix digest "
          f"{MATRIX_DIGEST[branch]}  features {EXPECTED['features'][branch]:,}")
require("the feature contract carries no timestamp or session path",
        "/kaggle" not in json.dumps(FEATURE_CONTRACT)
        and "generated_at" not in json.dumps(FEATURE_CONTRACT), "stable")
require("the feature contract id is reproducible",
        R.feature_manifest_id(json.loads(json.dumps(FEATURE_CONTRACT, default=str)))
        == FEATURE_CONTRACT_ID, FEATURE_CONTRACT_ID)
require("the contract records the dimensionality confound as representation-level",
        "representation-level confound"
        in FEATURE_CONTRACT["interpretation_caveats"]["dimensionality_confound"],
        "recorded")
require("the contract pre-registers matched dimensionality as secondary only",
        "SECONDARY / SENSITIVITY"
        in FEATURE_CONTRACT["interpretation_caveats"]["primary_comparison"],
        "never replaces the primary comparison")
require("the contract records the time-metadata gate verdict",
        FEATURE_CONTRACT["time_base"]["verdict"] == "PASS",
        FEATURE_CONTRACT["time_base"]["verdict"])
require("the contract records the physical-time scaling policy",
        FEATURE_CONTRACT["time_base"]["scaling"]["velocity"].endswith("f^1")
        and FEATURE_CONTRACT["time_base"]["scaling"]["acceleration"].endswith("f^2")
        and FEATURE_CONTRACT["time_base"]["scaling"]["duration_seconds"] == "T / f",
        "f^1 / f^2 / T over f")
require("the contract states that fps is never a predictor",
        "never itself an RF predictor" in FEATURE_CONTRACT["time_base"]["policy"],
        "auxiliary metadata only")
require("the contract records heterogeneous fps as expected, not a failure",
        "not a failure condition" in FEATURE_CONTRACT["time_base"]["heterogeneity"],
        "two regimes handled")
FORMULAS = FEATURE_CONTRACT["formulas"]
require("the recorded velocity formula carries the factor f",
        "(x[t+1] - x[t]) * f" in FORMULAS["velocity"]
        and "per second" in FORMULAS["velocity"]
        and "per frame" not in FORMULAS["velocity"], "physical-time definition")
require("the recorded acceleration formula carries the factor f^2",
        "(x[t+2] - 2*x[t+1] + x[t]) * f^2" in FORMULAS["acceleration"]
        and "per second^2" in FORMULAS["acceleration"], "physical-time definition")
require("the recorded mean 3D speed formula carries the factor f",
        FORMULAS["mean_3d_speed"].startswith("per LANDMARK: f * (1/(T-1))"),
        "physical-time definition")
require("the recorded duration formula is T / f",
        FORMULAS["duration_seconds"].startswith("T / f"), "T / f")
require("no formula claims that no frame rate is applied",
        not any("no frame rate is applied" in str(v) for v in FORMULAS.values()),
        "stale wording absent")
require("the formulas agree with the time_base scaling block",
        FEATURE_CONTRACT["time_base"]["scaling"]["velocity"].endswith("f^1")
        and "* f" in FORMULAS["velocity"]
        and FEATURE_CONTRACT["time_base"]["scaling"]["acceleration"].endswith("f^2")
        and "f^2" in FORMULAS["acceleration"],
        "one internally consistent definition")
require("positional statistics carry no absolute independence claim",
        "may retain small sampling-density effects" in FORMULAS["positional_statistics"]
        and "frame-rate independent" not in json.dumps(FEATURE_CONTRACT),
        "no absolute claim")
require("mean 3D speed is described as FPS-corrected, not frame-rate independent",
        "FPS-corrected" in FORMULAS["mean_3d_speed"]
        and "variable-frame-rate limitation" in FORMULAS["mean_3d_speed"],
        "subject to the VFR limitation")
require("the contract binds the time metadata id",
        FEATURE_CONTRACT["time_metadata_provenance"]["time_metadata_id"]
        == TIME_METADATA_ID, TIME_METADATA_ID)
require("the contract records the Phase 3 source file hashes",
        set(FEATURE_CONTRACT["time_metadata_provenance"]["source_files"])
        == set(SOURCE_SHA256) and all(len(v) == 64 for v in SOURCE_SHA256.values()),
        f"{len(SOURCE_SHA256)} file(s), sha256")
require("the contract records the bound clip count",
        FEATURE_CONTRACT["time_metadata_provenance"]["clip_count"] == EXPECTED["clips"],
        f"{EXPECTED['clips']:,}")
require("the contract records the frame-rate validation rule",
        FEATURE_CONTRACT["time_metadata_provenance"]["validation_rule"] == GATE["rule"],
        "recorded")
require("the contract records the CAP_PROP_FPS limitation",
        "variable frame timing" in FEATURE_CONTRACT["time_base"]["limitation"],
        "stated limitation")
require("the contract records the zero-variance rule without a survival assumption",
        "may be dropped" in FEATURE_CONTRACT["zero_variance_filter"]["scope"]
        and "no minimum kept count"
        in FEATURE_CONTRACT["zero_variance_filter"]["scope"], "recorded")
require("the contract records the interpolation caveat",
        "interpolation behaviour"
        in FEATURE_CONTRACT["interpretation_caveats"]["interpolation"], "recorded")

# ---- B. hash the published module -----------------------------------------
RF_MODULE_SHA256 = R.file_sha256(RF_MODULE_PATH)
print(f"\nPUBLISHED MODULE\n  {RF_MODULE_PATH.name}\n  sha256 {RF_MODULE_SHA256}")
require("the published feature module is byte-identical to what was tested",
        RF_MODULE_PATH.read_text(encoding="utf-8") == RF_MODULE_SOURCE,
        f"{RF_MODULE_PATH.stat().st_size:,} bytes")
require("the feature module hash recomputes from disk",
        R.file_sha256(RF_MODULE_PATH) == RF_MODULE_SHA256, RF_MODULE_SHA256[:16] + "...")

MODULE_BLOCK = {"filename": RF_MODULE_PATH.name, "sha256": RF_MODULE_SHA256,
                "sha256_short": RF_MODULE_SHA256[:16],
                "bytes": RF_MODULE_PATH.stat().st_size,
                "lines": len(RF_MODULE_SOURCE.splitlines()),
                "feature_spec_version": FEATURE_SPEC_VERSION,
                "verify_before_import":
                    "hashlib.sha256(open(path, 'rb').read()).hexdigest()"}

# ---- C. the manifest -------------------------------------------------------
RUN_METADATA = {
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "environment": ENVIRONMENT,
    "notebook8_dir": str(NB8_DIR),
    "resolved_paths_this_session": context.provenance["resolved"],
    "note": "recorded for traceability and deliberately excluded from "
            "feature_contract_id, which must not change between reruns",
}
MANIFEST = {
    "feature_contract_id": FEATURE_CONTRACT_ID,
    "feature_contract": FEATURE_CONTRACT,
    "filter_set_id": FILTER_SET_ID,
    "filter_digests": FILTER_DIGESTS,
    "module": MODULE_BLOCK,
    "matrices": {branch: {"file": MATRIX_PATH[branch].name,
                          "shape": list(MATRIX_SHAPE[branch]),
                          "dtype": "float32", "digest": MATRIX_DIGEST[branch],
                          "bytes": MATRIX_PATH[branch].stat().st_size,
                          "row_order": CLIP_ORDER_PATH.name,
                          "storage": "uncompressed .npy so notebook 10 can memory-map it"}
                 for branch in R.BRANCHES},
    "specs": {branch: {"file": SPEC_PATH[branch].name, "hash": SPEC_HASH[branch],
                       "rows": int(EXPECTED["features"][branch])}
              for branch in R.BRANCHES},
    "filters": {"folds": {b: FOLD_FILTER_PATH[b].name for b in R.BRANCHES},
                "control": {b: CONTROL_FILTER_PATH[b].name for b in R.BRANCHES},
                "summary": filter_table.to_dict("records")},
    "constant_columns": CONSTANT_REPORT,
    "time_metadata_id": TIME_METADATA_ID,
    "time_metadata_provenance": TIME_METADATA_PROVENANCE,
    "fps_audit": FPS_AUDIT,
    "run_metadata": RUN_METADATA,
}


def atomic_write(path: Path, writer) -> None:
    temporary = path.parent / (path.name + ".tmp")
    with open(temporary, "wb") as handle:
        writer(handle)
    os.replace(temporary, path)


atomic_write(FEATURE_MANIFEST, lambda h: h.write(
    json.dumps(MANIFEST, indent=2, sort_keys=True, default=str).encode()))
reloaded_manifest = json.loads(FEATURE_MANIFEST.read_text(encoding="utf-8"))
require("feature contract id reproduces from the manifest on disk",
        R.feature_manifest_id(reloaded_manifest["feature_contract"]) == FEATURE_CONTRACT_ID
        == reloaded_manifest["feature_contract_id"], FEATURE_CONTRACT_ID)
require("filter set id reproduces from the manifest on disk",
        R.feature_manifest_id(reloaded_manifest["filter_digests"]) == FILTER_SET_ID
        == reloaded_manifest["filter_set_id"], FILTER_SET_ID)
require("the manifest records both matrix digests",
        all(reloaded_manifest["matrices"][b]["digest"] == MATRIX_DIGEST[b]
            for b in R.BRANCHES), "recorded")
require("run metadata is stored outside the hashed contract",
        "generated_at_utc" not in json.dumps(reloaded_manifest["feature_contract"]),
        "separated")
persisted_fps = pd.read_csv(FPS_TABLE_PATH, dtype={"clip_id": str, "fps_hex": str})
reloaded_fps = {row.clip_id: float.fromhex(row.fps_hex)
                for row in persisted_fps.itertuples()}
require("time metadata id reproduces from the persisted table and the manifest",
        R.time_metadata_id(reloaded_fps, CLIP_IDS) == TIME_METADATA_ID
        == reloaded_manifest["time_metadata_id"]
        == reloaded_manifest["feature_contract"]["time_metadata_provenance"][
            "time_metadata_id"], TIME_METADATA_ID)
require("the manifest records the Phase 3 source hashes",
        reloaded_manifest["feature_contract"]["time_metadata_provenance"]["source_files"]
        == SOURCE_SHA256, f"{len(SOURCE_SHA256)} file(s)")
require("the manifest names the six constant columns explicitly",
        len(reloaded_manifest["constant_columns"]) == 2 * EXPECTED["constant_base_columns"],
        f"{len(reloaded_manifest['constant_columns'])} rows across both branches")

# ---- D. final gate, then the report ---------------------------------------
FINAL_FAILURES = failures()
SKIPPED = [entry for entry in CHECKS if entry["status"] == "skip"]
VERDICT = "FAIL" if FINAL_FAILURES else "PASS"

report = {
    "notebook": "9-random-forest-feature-engineering",
    "feature_spec_version": FEATURE_SPEC_VERSION, "verdict": VERDICT,
    "feature_contract_id": FEATURE_CONTRACT_ID, "filter_set_id": FILTER_SET_ID,
    "time_metadata_id": TIME_METADATA_ID,
    "experiment_id": REQUIRED_EXPERIMENT_ID,
    "loader_contract_id": REQUIRED_LOADER_CONTRACT_ID,
    "module": MODULE_BLOCK,
    "checks_total": len(CHECKS),
    "checks_passed": len([c for c in CHECKS if c["status"] == "pass"]),
    "checks_failed": len(FINAL_FAILURES), "checks_skipped": len(SKIPPED),
    "checks": CHECKS,
    "dimensions": {b: int(EXPECTED["features"][b]) for b in R.BRANCHES},
    "matrix_digests": MATRIX_DIGEST, "spec_hashes": SPEC_HASH,
    "filter_summary": filter_table.to_dict("records"),
    "additional_dropped_features": EXTRA_DROPPED,
    "fps_audit": FPS_AUDIT,
    "constant_columns": CONSTANT_REPORT,
    "runtime_seconds": round(time.time() - STARTED, 1),
    "run_metadata": RUN_METADATA,
}
atomic_write(REPORT_JSON, lambda h: h.write(json.dumps(report, indent=2, default=str).encode()))
atomic_write(REPORT_CSV, lambda h: pd.DataFrame(CHECKS)[
    ["check", "status", "passed", "detail"]].to_csv(h, index=False))
print(f"\nwrote {REPORT_JSON.name} and {REPORT_CSV.name} "
      f"({len(CHECKS)} checks, verdict {VERDICT})")

if FINAL_FAILURES:
    for entry in FINAL_FAILURES:
        print(f"  FAIL  {entry['check']}: {entry['detail']}")
    raise RuntimeError(
        f"{len(FINAL_FAILURES)} checks failed. The validation report was written for "
        "diagnosis, but the feature set is NOT usable and notebook 10 must not read it.")

written = [FEATURE_MANIFEST, RF_MODULE_PATH, *MATRIX_PATH.values(), *SPEC_PATH.values(),
           CLIP_ORDER_PATH, *FOLD_FILTER_PATH.values(), *CONTROL_FILTER_PATH.values(),
           FPS_AUDIT_PATH, FPS_TABLE_PATH, REPORT_JSON, REPORT_CSV]
total_bytes = sum(p.stat().st_size for p in written)

print("=" * 78)
print("NOTEBOOK 9 COMPLETE - FEATURE SPECIFICATION FROZEN")
print("=" * 78)
print(f"  experiment id        : {REQUIRED_EXPERIMENT_ID}")
print(f"  loader contract id   : {REQUIRED_LOADER_CONTRACT_ID}")
print(f"  feature contract id  : {FEATURE_CONTRACT_ID}")
print(f"  filter set id        : {FILTER_SET_ID}")
print(f"  time metadata id     : {TIME_METADATA_ID}")
print(f"  feature module sha256: {RF_MODULE_SHA256}")
print(f"  features             : full {EXPECTED['features']['full']:,} | "
      f"reduced {EXPECTED['features']['reduced']:,} "
      f"(ratio {EXPECTED['features']['full'] / EXPECTED['features']['reduced']:.2f})")
print(f"  checks               : {report['checks_passed']} passed, "
      f"{report['checks_failed']} failed, {report['checks_skipped']} skipped "
      f"of {len(CHECKS)}")
print(f"  runtime              : {(time.time() - STARTED) / 60:.1f} min")
print(f"  outputs              : {total_bytes / 1e6:.0f} MB")

print("\n  Files written:")
for path in written:
    print(f"    {path.name:<36} {path.stat().st_size:>13,} bytes")

print("\n  Frozen here: the feature formulas, the feature-index map, both feature")
print("  matrices, and the per-fold and control zero-variance keep-masks.")
print("\n  NOT frozen here: Random Forest hyperparameters. Those belong to notebook 10")
print("  and must be frozen there before any test evaluation, identical between the")
print("  full and reduced branches.")
print("\n  Carried forward as an interpretation constraint: the 2.80x feature-")
print("  dimensionality gap is a representation-level confound; any matched-")
print("  dimensionality analysis is secondary and never replaces the native comparison.")

print("\n  Notebook 10 starts with:")
print("    import hashlib, json, sys")
print("    root = '<attached notebook 9 output>'")
print("    manifest = json.load(open(f'{root}/rf_feature_manifest.json'))")
print("    digest = hashlib.sha256("
      "open(f'{root}/bsl_rf_features.py', 'rb').read()).hexdigest()")
print("    assert digest == manifest['module']['sha256']")
print("    sys.path.insert(0, root); import bsl_rf_features as R")
print("    matrix = np.load(f'{root}/rf_features_{branch}.npy', mmap_mode='r')")
print("    keep   = R.require_filter_scope(")
print("        R.load_filters(f'{root}/fold_feature_filters_{branch}.npz')[f'fold_{fold}'],")
print("        branch, 'fold', fold)")

print("\n" + "=" * 78)
print(f"  Quote feature contract id {FEATURE_CONTRACT_ID} and filter set id "
      f"{FILTER_SET_ID}")
print("  in notebooks 10 and 13.")
print("  STOP. Do not build notebook 10 until this real Kaggle log has been reviewed.")
print("=" * 78)